# Major CS — V5 com força por mapa e previsão do veto

A V5 **adiciona duas entradas** às **13 entradas da V4**. Mantemos o histórico, o ranking HLTV, a forma individual, o Elo dos adversários e o ajuste de tiers. Usamos a mesma divisão cronológica e o mesmo simulador suíço.

1. Reproduzir as entradas e o treino da V4.
2. Auditar a base de mapas, estimar o Elo de cada equipe por mapa e prever a saída do veto.
3. Agregar as chances por sequência, treinar a V5 e comparar o teste com a V4.
4. Simular 100.000 Majors e mostrar as chances por fase e o caminho do favorito ao título.

Execute as células em ordem no ambiente Python do projeto. Quando necessário, indique a pasta de dados com `MAJOR_CS_DADOS`.

## 1. Ambiente e leitura

O CSV mistura séries completas com mapas individuais. Carregamos o identificador do confronto, o placar, o formato, o horário e os jogadores. As estatísticas produzidas durante o jogo ficam fora das entradas do modelo. Os CSVs por tier servem para identificar os grupos de avaliação e reproduzir o histórico da referência.

In [1]:
import hashlib
import json
import os
import sys
from pathlib import Path, PureWindowsPath
from tempfile import TemporaryDirectory
from unittest.mock import patch

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score

candidatas = [Path.cwd(), Path.home() / "Desktop" / "Major" / "Predict-do-major"]
if os.environ.get("MAJOR_CS_DADOS"):
    candidatas.insert(0, Path(os.environ["MAJOR_CS_DADOS"]).expanduser())

PASTA_DADOS = next(
    (pasta.resolve() for pasta in candidatas if (pasta / "cs2_all_tiers_games.csv").is_file()),
    None,
)
if PASTA_DADOS is None:
    raise FileNotFoundError("Indique a pasta Predict-do-major em MAJOR_CS_DADOS.")

# Os módulos originais são usados como dependência local, sem gerar cache na pasta fonte.
pasta_scripts = str(PASTA_DADOS / "scripts")
if pasta_scripts not in sys.path:
    sys.path.insert(0, pasta_scripts)
with patch.object(sys, "dont_write_bytecode", True):
    import reconstruir_major as reconstrucao
    import simular_singapura as simulador
    from modelo_simetrico import ModeloSimetrico

COLUNAS_JOGADORES = [
    f"team{lado}_player{numero}" for lado in (1, 2) for numero in range(1, 6)
]
COLUNAS_DADOS = [
    "match_id", "game_id", "tournament", "datetime", "is_total",
    "team1", "team2", "bestOf", "score1_match", "score2_match", "team1_win",
] + COLUNAS_JOGADORES

caminho_csv = PASTA_DADOS / "cs2_all_tiers_games.csv"
caminho_tier1 = PASTA_DADOS / "cs2_tier1_games.csv"
jogos = pd.read_csv(caminho_csv, usecols=COLUNAS_DADOS, parse_dates=["datetime"])
jogos_tier1 = pd.read_csv(caminho_tier1, usecols=COLUNAS_DADOS, parse_dates=["datetime"])

print(f"Python {sys.version.split()[0]} · pandas {pd.__version__} · scikit-learn {sklearn.__version__}")
display(pd.DataFrame([
    {"base": "Todos os tiers", "linhas": len(jogos),
     "totais de séries": jogos["is_total"].sum(), "mapas": jogos["is_total"].eq(False).sum()},
    {"base": "Tier 1", "linhas": len(jogos_tier1),
     "totais de séries": jogos_tier1["is_total"].sum(), "mapas": jogos_tier1["is_total"].eq(False).sum()},
]))

Python 3.15.2 · pandas 3.1.1 · scikit-learn 1.9.1


,base,linhas,totais de séries,mapas
0,Todos os tiers,24109,11482,12627
1,Tier 1,13044,5194,7850


## 2. Limpeza e alvo

### 2.1 Uma linha por série

`is_total` identifica a linha da série. O `match_id` identifica o confronto; os IDs de equipe do CSV variam de partida para partida, como vimos no notebook original. Mantemos a chave de histórico pelo nome exato da equipe e inspecionamos as repetições de `match_id` antes de acumular os jogos.

In [2]:
totais = jogos.loc[jogos["is_total"].eq(True)].copy()
duplicadas = totais.loc[totais["match_id"].duplicated(keep=False)].copy()
duplicadas["jogadores_preenchidos"] = duplicadas[COLUNAS_JOGADORES].notna().sum(axis=1)
display(duplicadas.sort_values(["match_id", "game_id"])[[
    "match_id", "game_id", "tournament", "team1", "team2", "datetime",
    "score1_match", "score2_match", "team1_win", "jogadores_preenchidos",
]])

,match_id,game_id,tournament,team1,team2,datetime,score1_match,score2_match,team1_win,jogadores_preenchidos
5061,7287899,-153225.0,PGL Astana 2026. Закрытая европейская квалифик...,K27,WW Team,2026-03-31 14:00:00,2.0,0.0,1.0,10
5060,7287899,-153223.0,PGL Astana 2026. Closed European Qualifier,K27,WW Team,2026-03-31 14:00:00,2.0,0.0,0.0,10
5100,7298778,-153207.0,Roman Imperium Cup VII,G2 Esports,BETBOOM TEAM,2026-03-30 20:10:00,1.0,2.0,0.0,10
5099,7298778,-153205.0,Roman Imperium Cup VII,G2 Esports,BetBoom Team,2026-03-30 20:10:00,1.0,2.0,0.0,10
5081,7298808,-153218.0,Roman Imperium Cup VII,Acend,G2 Esports,2026-03-31 00:40:00,0.0,2.0,0.0,10
5080,7298808,-153217.0,Roman Imperium Cup VII,Acend,G2 Esports,2026-03-31 00:40:00,0.0,2.0,0.0,9
4964,7298940,-153267.0,PGL Astana 2026. Закрытая европейская квалифик...,Lavked,100 Thieves,2026-04-02 11:00:00,1.0,2.0,0.0,10
4963,7298940,-153266.0,PGL Astana 2026. Closed European Qualifier,Lavked,100 Thieves,2026-04-02 11:00:00,1.0,2.0,0.0,10


### 2.2 Resolver os quatro duplicados e construir o alvo

- Para `7287899` e `7298940`, mantemos a mesma linha retida pelo notebook original. São repetições da qualificação europeia do PGL Astana.
- Para `7298778`, as escalações coincidem e varia a grafia de BetBoom. Mantemos a grafia `BetBoom Team` nesse registro.
- Para `7298808`, mantemos a linha com a escalação mais completa.

O alvo vale 1 quando `score1_match > score2_match`, e 0 quando o segundo time vence. Retiramos séries sem placar decisivo ou sem identificação completa. Mantemos `bestOf` vazio quando o formato está ausente. A imputação será ajustada só com o treino.

In [3]:
TOTAIS_DESCARTADOS = {
    7287899: -153223,
    7298940: -153266,
    7298778: -153207,
    7298808: -153217,
}
ALVO = "team1_win_corrigido"

def limpar_series(base):
    series = base.loc[base["is_total"].eq(True)].copy()
    duplicatas = series.loc[series["match_id"].duplicated(keep=False)]
    for confronto, par in duplicatas.groupby("match_id"):
        comparaveis = par[["datetime", "bestOf", "score1_match", "score2_match"]].copy()
        comparaveis["team1"] = par["team1"].str.casefold()
        comparaveis["team2"] = par["team2"].str.casefold()
        descarte = par["game_id"].eq(TOTAIS_DESCARTADOS.get(confronto))
        if len(par) != 2 or len(comparaveis.drop_duplicates()) != 1 or descarte.sum() != 1:
            raise ValueError(f"Revise a duplicata da série {confronto}.")
        series = series.drop(index=par.index[descarte])

    assert series["match_id"].is_unique, "Cada série precisa ter um identificador único."
    invalidas = (
        series[["match_id", "datetime", "team1", "team2", "score1_match", "score2_match"]]
        .isna().any(axis=1)
        | series["team1"].eq(series["team2"])
        | series["score1_match"].eq(series["score2_match"])
    )
    rejeitadas = series.loc[invalidas].copy()
    series = series.loc[~invalidas].sort_values(["datetime", "match_id"]).reset_index(drop=True)
    series[ALVO] = series["score1_match"].gt(series["score2_match"]).astype(int)
    return series, rejeitadas

series_v3, rejeitadas = limpar_series(jogos)
series_tier1, rejeitadas_tier1 = limpar_series(jogos_tier1)
assert set(series_tier1["match_id"]).issubset(series_v3["match_id"])
print(f"Séries aproveitadas · todos os tiers {len(series_v3):,} · Tier 1 {len(series_tier1):,}")
print(f"Séries sem formato informado: {series_v3['bestOf'].isna().sum()}")
print(f"Indicadores originais divergentes do placar: {(series_v3['team1_win'].notna() & series_v3['team1_win'].ne(series_v3[ALVO])).sum()}")
display(rejeitadas[["match_id", "team1", "team2", "score1_match", "score2_match"]])

Séries aproveitadas · todos os tiers 11,471 · Tier 1 5,190
Séries sem formato informado: 119
Indicadores originais divergentes do placar: 4891


,match_id,team1,team2,score1_match,score2_match
3117,10095505,США,Канада,NaN,NaN
4771,7299208,FaZe Clan,CYBERSHOKE Esports,1.0,1.0
4830,7298927,NRG Esports,Team Voca,1.0,1.0
4881,7299053,G2 Esports,GamerLegion,1.0,1.0
5067,7298791,Passion UA,FaZe Clan,2.0,2.0
16122,1629909,ALTERNATE aTTaX,THE SUSPECT,1.0,1.0
23528,10064713,NaN,NaN,NaN,NaN


### 2.3 Etiquetar o tier para a avaliação

Cruzamos `match_id` com os três arquivos por tier. A etiqueta se refere à competição registrada no dataset e serve só para diagnóstico. Séries presentes exclusivamente no CSV completo continuam na base com a etiqueta **Sem tier identificado**.

In [4]:
ids_por_tier = {}
for numero in (1, 2, 3):
    dados_tier = pd.read_csv(
        PASTA_DADOS / f"cs2_tier{numero}_games.csv", usecols=["match_id", "is_total"]
    )
    ids_por_tier[f"Tier {numero}"] = set(dados_tier.loc[dados_tier["is_total"], "match_id"])

tier_por_id = {confronto: tier for tier, ids in ids_por_tier.items() for confronto in ids}
assert len(tier_por_id) == sum(map(len, ids_por_tier.values())), "Há IDs em mais de um tier."
series_v3["tier"] = series_v3["match_id"].map(tier_por_id).fillna("Sem tier identificado")
display(series_v3.groupby("tier").agg(
    series=("match_id", "size"), de=("datetime", "min"), ate=("datetime", "max")
))

,series,de,ate
tier,,,
Sem tier identificado,22,2026-11-12 13:00:00,2026-11-14 21:30:00
Tier 1,5190,2023-01-10 09:30:00,2026-11-14 13:30:00
Tier 2,5352,2023-10-05 20:00:00,2026-11-01 22:00:00
Tier 3,907,2023-10-11 16:00:00,2026-11-01 16:00:00


## 3. Histórico disponível antes de cada jogo

### 3.1 Uma perspectiva por equipe

Cada série vira duas participações. Se A venceu, A recebe uma vitória e B recebe uma derrota. Somamos participações e vitórias por nome e horário antes de acumular. Assim, duas partidas com o mesmo horário ficam juntas no histórico e só ficam disponíveis para previsões posteriores.

In [5]:
def acumular_historico(series):
    participacoes = pd.concat([
        series[["datetime", "team1", ALVO]].rename(columns={"team1": "equipe", ALVO: "venceu"}),
        series[["datetime", "team2", ALVO]].rename(columns={"team2": "equipe", ALVO: "venceu"})
        .assign(venceu=lambda tabela: 1 - tabela["venceu"]),
    ], ignore_index=True)
    blocos = (
        participacoes.groupby(["equipe", "datetime"], as_index=False)
        .agg(jogos_no_horario=("venceu", "size"), vitorias_no_horario=("venceu", "sum"))
        .sort_values("datetime")
    )
    por_equipe = blocos.groupby("equipe", sort=False)
    blocos["jogos_anteriores"] = por_equipe["jogos_no_horario"].cumsum()
    blocos["vitorias_anteriores"] = por_equipe["vitorias_no_horario"].cumsum()
    return blocos

historico_v3 = acumular_historico(series_v3)
historico_tier1 = acumular_historico(series_tier1)
assert historico_v3["jogos_no_horario"].sum() == 2 * len(series_v3)
assert historico_v3["vitorias_no_horario"].sum() == len(series_v3)
display(historico_v3.loc[historico_v3["equipe"].eq("100 Thieves")].head(5))

,equipe,datetime,jogos_no_horario,vitorias_no_horario,jogos_anteriores,vitorias_anteriores
19,100 Thieves,2026-02-20 20:10:00,1,0,1,0
20,100 Thieves,2026-02-20 21:30:00,1,1,2,1
21,100 Thieves,2026-02-21 00:30:00,1,1,3,2
22,100 Thieves,2026-02-21 17:30:00,1,0,4,2
23,100 Thieves,2026-03-31 17:00:00,1,0,5,2


### 3.2 Ligar cada confronto ao último bloco estritamente anterior

Para cada lado, buscamos o último acumulado com horário menor que o do jogo. Uma estreia começa com zero jogos, zero vitórias e taxa ausente. A referência consulta só o histórico Tier 1, inclusive quando é usada para prever uma partida de Tier 2 ou 3. A V3 consulta o histórico completo para os mesmos confrontos.

In [6]:
CONTAGENS = ["jogos_anteriores", "vitorias_anteriores"]
ATRIBUTOS_HISTORICO = CONTAGENS + ["taxa_vitorias_anteriores"]

def juntar_historico(series, historico):
    saida = series.copy()
    consulta = historico[["equipe", "datetime"] + CONTAGENS].rename(
        columns={"datetime": "horario_historico"}
    )
    for lado in (1, 2):
        time = f"team{lado}"
        atual = series[["match_id", "datetime", time]].rename(columns={time: "equipe"})
        anterior = pd.merge_asof(
            atual.sort_values("datetime"), consulta,
            left_on="datetime", right_on="horario_historico", by="equipe",
            direction="backward", allow_exact_matches=False,
        )
        tem_historico = anterior["horario_historico"].notna()
        assert anterior.loc[tem_historico, "horario_historico"].lt(
            anterior.loc[tem_historico, "datetime"]
        ).all()
        anterior[CONTAGENS] = anterior[CONTAGENS].fillna(0).astype(int)
        anterior["taxa_vitorias_anteriores"] = (
            anterior["vitorias_anteriores"] / anterior["jogos_anteriores"].replace(0, np.nan)
        )
        colunas = {atributo: f"{time}_{atributo}" for atributo in ATRIBUTOS_HISTORICO}
        saida = saida.merge(
            anterior[["match_id"] + ATRIBUTOS_HISTORICO].rename(columns=colunas),
            on="match_id", validate="one_to_one",
        )
    for atributo in ATRIBUTOS_HISTORICO:
        saida[f"dif_{atributo}"] = saida[f"team1_{atributo}"] - saida[f"team2_{atributo}"]
    return saida

dados_v3 = juntar_historico(series_v3, historico_v3)
dados_ref = juntar_historico(series_v3, historico_tier1)
assert (dados_v3["team1_jogos_anteriores"] >= dados_ref["team1_jogos_anteriores"]).all()
assert (dados_v3["team2_jogos_anteriores"] >= dados_ref["team2_jogos_anteriores"]).all()
display(dados_v3[["datetime", "team1", "team2", *reconstrucao.V1_COLUMNS, ALVO]].tail(5))

,datetime,team1,team2,dif_jogos_anteriores,dif_vitorias_anteriores,dif_taxa_vitorias_anteriores,bestOf,team1_win_corrigido
11466,2026-11-14 16:15:00,Team Buster,Team WorldEdit,27,16,-0.025000,3.0,1
11467,2026-11-14 16:15:00,Team leva2k,Team NS,-1,0,0.500000,3.0,1
11468,2026-11-14 18:00:00,Iberian Soul,FORZE Reload,-1,5,0.087814,3.0,1
11469,2026-11-14 19:00:00,Team PCH3LK1N,Team Jerry,2,-2,-0.317460,3.0,1
11470,2026-11-14 21:30:00,WW Team,Hype Tech,63,33,NaN,3.0,0


## 4. Ranking HLTV e parâmetros da referência

### 4.1 Pontos conhecidos antes da partida

Reusamos a regra do projeto. A edição precisa ter sido publicada antes do dia do confronto, ter no máximo 14 dias e combinar nome com pelo menos três jogadores em comum. A diferença de pontos fica ausente se um dos lados não puder ser confirmado. A validação histórica usa a escalação registrada no CSV. A simulação usa a escalação publicada para o Major. As equivalências intermediárias são produzidas em pasta temporária.

In [7]:
with TemporaryDirectory(prefix="major-v3-ranking-") as pasta_temporaria:
    with (
        patch.object(reconstrucao, "PROJECT", PASTA_DADOS),
        patch.object(reconstrucao, "OUTPUT", Path(pasta_temporaria)),
    ):
        atributos_ranking, auditoria_ranking = reconstrucao.ranking_features(series_v3)

assert len(atributos_ranking) == len(series_v3)
pontuacao = atributos_ranking[["match_id", "dif_pontos_hltv"]]
dados_v3 = dados_v3.merge(pontuacao, on="match_id", validate="one_to_one")
dados_ref = dados_ref.merge(pontuacao, on="match_id", validate="one_to_one")

display(dados_v3.groupby("tier")["dif_pontos_hltv"].agg(
    series="size", ranking_dois_lados="count", cobertura=lambda pontos: pontos.notna().mean()
).round(3))

,series,ranking_dois_lados,cobertura
tier,,,
Sem tier identificado,22,1,0.045
Tier 1,5190,3446,0.664
Tier 2,5352,1609,0.301
Tier 3,907,141,0.155


### 4.2 Conferir a reprodução da versão anterior

O manifesto salvo informa quais atributos foram usados. Conferimos cada um deles, mais o alvo, nas séries Tier 1 do experimento anterior. Essa verificação testa se a V3 poderá ser comparada à referência sem mudar a definição das entradas.

In [8]:
relatorio_ref = json.loads(
    (PASTA_DADOS / "dados_externos" / "reconstrucao" / "ultimo_experimento.json").read_bytes()
)
pasta_experimento = PASTA_DADOS / "experimentos" / PureWindowsPath(
    relatorio_ref["pasta_experimento"]
).name
selecao_ref = json.loads((pasta_experimento / "selecao_antes_do_teste.json").read_bytes())
ENTRADAS = selecao_ref["entradas"]
assert selecao_ref["modelo"] == "V1 + ranking (simetrico)"
assert ENTRADAS == reconstrucao.V1_COLUMNS + ["dif_pontos_hltv"]

colunas_comparadas = ENTRADAS + [ALVO]
base_ref_salva = pd.read_csv(
    pasta_experimento / "base_experimentos.csv", usecols=["match_id", *colunas_comparadas]
).set_index("match_id").sort_index()[colunas_comparadas]
base_ref_recalculada = dados_ref.loc[dados_ref["tier"].eq("Tier 1")].set_index("match_id")
pd.testing.assert_frame_equal(
    base_ref_recalculada.sort_index()[colunas_comparadas], base_ref_salva,
    check_dtype=False, check_exact=False, rtol=1e-10, atol=1e-12,
)
modelo_ref = joblib.load(pasta_experimento / "modelo_selecionado.joblib")
assert list(modelo_ref.modelo.feature_names_in_) == ENTRADAS
print(f"{len(base_ref_salva):,} séries Tier 1 conferidas atributo por atributo.")
print(ENTRADAS)

5,190 séries Tier 1 conferidas atributo por atributo.
['dif_jogos_anteriores', 'dif_vitorias_anteriores', 'dif_taxa_vitorias_anteriores', 'bestOf', 'dif_pontos_hltv']


## 5. Separação cronológica e treino

### 5.1 Manter os cortes do primeiro notebook

| Conjunto | Período | Uso |
|---|---|---|
| Treino | Antes de 01/01/2026 | Ajustar os parâmetros e as transformações |
| Validação | De 01/01/2026 até 30/06/2026 | Comparar o comportamento das versões |
| Teste | A partir de 01/07/2026 | Medir o resultado fora do período de treino |

A avaliação usa o horário registrado no CSV como referência temporal. Uma série do teste pode consultar resultados de séries com horário anterior, inclusive no mesmo período. O modelo e o preenchimento de valores ausentes ficam congelados durante a avaliação.

In [9]:
INICIO_VALIDACAO = pd.Timestamp("2026-01-01")
INICIO_TESTE = pd.Timestamp("2026-07-01")
data = dados_v3["datetime"]
particoes = {
    "Treino": data.lt(INICIO_VALIDACAO),
    "Validação": data.ge(INICIO_VALIDACAO) & data.lt(INICIO_TESTE),
    "Teste": data.ge(INICIO_TESTE),
}
assert dados_v3["match_id"].equals(dados_ref["match_id"])
assert (sum(mascara.astype(int) for mascara in particoes.values()) == 1).all()
display(pd.DataFrame([
    {"conjunto": nome, "total": int(mascara.sum()),
     "Tier 1": int((mascara & dados_v3["tier"].eq("Tier 1")).sum()),
     "Tier 2 e 3": int((mascara & dados_v3["tier"].isin(["Tier 2", "Tier 3"])).sum()),
     "Sem tier identificado": int((mascara & dados_v3["tier"].eq("Sem tier identificado")).sum()),
     "de": data[mascara].min(), "até": data[mascara].max()}
    for nome, mascara in particoes.items()
]).set_index("conjunto"))

,total,Tier 1,Tier 2 e 3,Sem tier identificado,de,até
conjunto,,,,,,
Treino,8109,3775,4334,0,2023-01-10 09:30:00,2025-12-28 18:15:00
Validação,1844,642,1202,0,2026-01-03 08:00:00,2026-06-30 21:20:00
Teste,1518,773,723,22,2026-07-01 11:45:00,2026-11-14 21:30:00


### 5.2 Treinar a V3 com o mesmo pipeline

O pipeline preenche valores ausentes com a mediana e acrescenta indicadores de ausência, padroniza as entradas e treina a regressão logística com `max_iter=1000`. Usamos a mesma função de criação e conferimos os parâmetros contra o modelo salvo. A probabilidade final usa as duas ordens da partida, com `bestOf` fixo e diferenças invertidas.

In [10]:
treino = dados_v3.loc[particoes["Treino"]]
pipeline_v3 = reconstrucao.new_pipeline()
for nome, etapa in pipeline_v3.named_steps.items():
    parametros_ref = modelo_ref.modelo.named_steps[nome].get_params(deep=False)
    assert json.dumps(etapa.get_params(deep=False), sort_keys=True) == json.dumps(parametros_ref, sort_keys=True)

pipeline_v3.fit(treino[ENTRADAS], treino[ALVO])
modelo_v3 = ModeloSimetrico(pipeline_v3)

amostra = treino[ENTRADAS].head(30)
invertida = amostra.copy()
diferenças = [nome for nome in ENTRADAS if nome.startswith("dif_")]
invertida[diferenças] = -invertida[diferenças]
np.testing.assert_allclose(
    modelo_v3.predict_proba(amostra)[:, 1] + modelo_v3.predict_proba(invertida)[:, 1], 1.0,
)
print(f"V3 treinada com {len(treino):,} séries.")
display(pd.DataFrame({
    "atributo": pipeline_v3.named_steps["imputacao"].statistics_,
}, index=ENTRADAS).rename(columns={"atributo": "mediana no treino"}))

V3 treinada com 8,109 séries.


,mediana no treino
dif_jogos_anteriores,3.000000
dif_vitorias_anteriores,2.000000
dif_taxa_vitorias_anteriores,0.015385
bestOf,3.000000
dif_pontos_hltv,11.000000


## 6. Construir o histórico dos jogadores

### 6.1 Identidade e qualidade das estatísticas

Usamos o ID local do jogador nas séries históricas e a linha de total retida na limpeza. A participação mantém o histórico individual quando o jogador troca de equipe. IDs repetidos dentro da mesma série são removidos dessa série por ambiguidade. O saldo por mapa é `(kills − mortes) / mapas jogados`. ADR e KAST vêm das colunas do CSV.

In [11]:
# Uma participação por jogador e série
from collections import defaultdict

campos_individuais = ["id", "adr", "kast", "kills", "deaths"]
prefixos = [f"team{lado}_player{numero}" for lado in (1, 2) for numero in range(1, 6)]
colunas_individuais = [
    f"{prefixo}_{campo}" for prefixo in prefixos for campo in campos_individuais
]
totais_jogadores = pd.read_csv(
    caminho_csv,
    usecols=["match_id", "game_id", "is_total", "games_played", *colunas_individuais],
)
totais_jogadores = totais_jogadores.loc[totais_jogadores["is_total"].eq(True)]
series_com_jogadores = series_v3.merge(
    totais_jogadores.drop(columns="is_total"), on=["match_id", "game_id"], validate="one_to_one",
)
assert series_com_jogadores["match_id"].equals(series_v3["match_id"])

partes = []
for lado in (1, 2):
    for posicao in range(1, 6):
        prefixo = f"team{lado}_player{posicao}"
        colunas = {f"{prefixo}_{campo}": campo for campo in campos_individuais}
        colunas[prefixo] = "nick_csv"
        partes.append(
            series_com_jogadores[["match_id", "datetime", "tier", "games_played", *colunas]]
            .rename(columns=colunas)
            .assign(lado=lado, posicao=posicao)
        )

slots = pd.concat(partes, ignore_index=True)
# Um ID repetido no mesmo confronto é tratado como identidade ambígua nessa série.
ids_validos = slots.loc[slots["id"].notna()].copy()
conflitos = ids_validos.duplicated(["match_id", "id"], keep=False)
jogadores = ids_validos.loc[~conflitos].copy()
jogadores["id"] = jogadores["id"].astype("int64")
jogadores["dia"] = jogadores["datetime"].dt.normalize()
jogadores["saldo_mapa"] = (
    (jogadores["kills"] - jogadores["deaths"]) / jogadores["games_played"].where(lambda s: s.gt(0))
)
assert len(slots) == 10 * len(series_v3)

auditoria_jogadores = (
    slots.assign(id_observado=slots["id"].notna(), adr_observado=slots["adr"].notna())
    .groupby("tier")
    .agg(slots=("match_id", "size"), ids=("id_observado", "sum"), adr=("adr_observado", "sum"))
)
auditoria_jogadores["IDs por série"] = 10 * auditoria_jogadores["ids"] / auditoria_jogadores["slots"]
auditoria_jogadores["Valores de ADR por série"] = 10 * auditoria_jogadores["adr"] / auditoria_jogadores["slots"]
print(f"{jogadores['id'].nunique():,} IDs com histórico; {int(conflitos.sum())} participações ambíguas removidas.")
display(auditoria_jogadores[["IDs por série", "Valores de ADR por série"]].round(2))

1,569 IDs com histórico; 48 participações ambíguas removidas.


,IDs por série,Valores de ADR por série
tier,,
Sem tier identificado,6.95,0.86
Tier 1,9.32,7.03
Tier 2,8.77,3.61
Tier 3,6.97,2.30


### 6.2 Medir a dificuldade de cada calendário

O Elo dá uma referência de força às equipes que os jogadores representaram e enfrentaram. A nota começa em 1500, com ajuste de 24 pontos multiplicado pela diferença entre resultado e expectativa. Todas as partidas de um dia consultam as notas do início do dia.

Para distinguir os níveis de competição, usamos a exposição anterior dos jogadores a eventos de Tier 1, 2 e 3, codificados como 2, 1 e 0, e a proporção de participações no Tier 3. Esses dois valores formam um índice de exposição entre 0 e 1 por escalação. A diferença dos índices entra como um ajuste nas chances de vitória e troca de sinal ao inverter os times. O tier do próprio jogo a prever fica fora das entradas.

In [12]:
# Elo antes do dia do confronto e contexto do tier nas séries históricas
ELO_INICIAL = 1500.0
K_ELO = 24.0
NIVEL_POR_TIER = {"Tier 1": 2.0, "Tier 2": 1.0, "Tier 3": 0.0}


def elo_anterior(series):
    notas = {}
    registros = []
    base = series[["match_id", "datetime", "team1", "team2", ALVO]].sort_values("datetime")
    for _, dia in base.groupby(base["datetime"].dt.normalize(), sort=True):
        incrementos = defaultdict(float)
        colunas = ["match_id", "team1", "team2", ALVO]
        for confronto, equipe_a, equipe_b, venceu_a in dia[colunas].itertuples(index=False, name=None):
            nota_a = notas.get(equipe_a, ELO_INICIAL)
            nota_b = notas.get(equipe_b, ELO_INICIAL)
            esperado_a = 1.0 / (1.0 + 10.0 ** ((nota_b - nota_a) / 400.0))
            ajuste = K_ELO * (venceu_a - esperado_a)
            incrementos[equipe_a] += ajuste
            incrementos[equipe_b] -= ajuste
            registros.extend([
                (confronto, 1, nota_a, nota_b),
                (confronto, 2, nota_b, nota_a),
            ])
        # Atualizar no fim do dia mantém todas as previsões do dia fora dos seus resultados.
        for equipe, ajuste in incrementos.items():
            notas[equipe] = notas.get(equipe, ELO_INICIAL) + ajuste
    return pd.DataFrame(registros, columns=["match_id", "lado", "elo_equipe", "elo_rival"])


contexto = elo_anterior(series_v3)
assert len(contexto) == 2 * len(series_v3)
jogadores = jogadores.merge(contexto, how="left", on=["match_id", "lado"], validate="many_to_one")
jogadores["nivel_eventos"] = jogadores["tier"].map(NIVEL_POR_TIER)
jogadores["fracao_tier3"] = jogadores["nivel_eventos"].eq(0).astype(float).where(
    jogadores["nivel_eventos"].notna()
)
display(jogadores.groupby("tier").agg(
    participacoes=("id", "size"),
    elo_equipe_mediano=("elo_equipe", "median"),
    elo_rival_mediano=("elo_rival", "median"),
).round(1))

,participacoes,elo_equipe_mediano,elo_rival_mediano
tier,,,
Sem tier identificado,153,1500.0,1500.0
Tier 1,48351,1579.8,1578.9
Tier 2,46929,1524.0,1519.9
Tier 3,6323,1517.4,1512.0


### 6.3 Pesar a forma recente de cada jogador

Consultamos apenas dias anteriores ao confronto. A cada 90 dias, o peso de uma observação cai pela metade. Uma média exige o equivalente a duas séries observadas. No nível da escalação, cada jogador tem o mesmo peso e o atributo exige três jogadores com histórico. A média dos dois menores saldos é calculada quando há quatro ou mais jogadores com saldo conhecido.

O histórico contém lacunas, sobretudo nos tiers menores. Elas entram como valores ausentes e passam pela mesma imputação da V3, aprendida no treino. Na validação, a composição do time vem da escalação registrada no CSV.

In [13]:
# Forma recente consultada antes do confronto
MEIA_VIDA_DIAS = 90
MINIMO_SERIES_EQUIVALENTES = 2.0
MINIMO_JOGADORES = 3
ORIGEM_PESOS = jogadores["dia"].min()
METRICAS_INDIVIDUAIS = ["adr", "kast", "saldo_mapa"]
CONTEXTO_INDIVIDUAL = ["elo_equipe", "elo_rival", "nivel_eventos", "fracao_tier3"]
MEDIDAS = METRICAS_INDIVIDUAIS + CONTEXTO_INDIVIDUAL


def criar_estados_diarios(participacoes):
    pesos = np.exp2((participacoes["dia"] - ORIGEM_PESOS).dt.days / MEIA_VIDA_DIAS)
    por_dia = participacoes[["id", "dia"]].copy()
    for coluna in MEDIDAS:
        observado = participacoes[coluna].notna()
        por_dia[f"peso_{coluna}"] = pesos.where(observado, 0.0)
        por_dia[f"soma_{coluna}"] = participacoes[coluna].fillna(0.0) * pesos
    blocos = por_dia.groupby(["id", "dia"], as_index=False).sum().sort_values(["id", "dia"])
    medidas_acumuladas = list(por_dia.columns[2:])
    blocos[medidas_acumuladas] = blocos.groupby("id")[medidas_acumuladas].cumsum()
    return blocos.rename(columns={"dia": "dia_anterior"}).sort_values("dia_anterior")


estados_individuais = criar_estados_diarios(jogadores)


def consultar_jogadores(elencos, estados=estados_individuais):
    """Consulte os IDs conhecidos e preserve as vagas sem identificação."""
    conhecidos = elencos.loc[elencos["id"].notna()].copy()
    conhecidos["id"] = conhecidos["id"].astype("int64")
    anteriores = pd.merge_asof(
        conhecidos.sort_values("dia"), estados,
        left_on="dia", right_on="dia_anterior", by="id",
        direction="backward", allow_exact_matches=False,
    )
    observado = anteriores["dia_anterior"].notna()
    assert anteriores.loc[observado, "dia_anterior"].lt(anteriores.loc[observado, "dia"]).all()
    ajuste = np.exp2(-(anteriores["dia"] - ORIGEM_PESOS).dt.days / MEIA_VIDA_DIAS)
    forma = anteriores[["consulta_id", "dia_anterior"]].copy()
    for medida in MEDIDAS:
        peso = anteriores[f"peso_{medida}"]
        peso_efetivo = peso * ajuste
        forma[f"{medida}_recente"] = anteriores[f"soma_{medida}"] / peso.where(
            peso_efetivo.ge(MINIMO_SERIES_EQUIVALENTES)
        )
        if medida == "adr":
            forma["series_equivalentes_adr"] = peso_efetivo
    return elencos.merge(forma, on="consulta_id", how="left", validate="one_to_one")


def resumir_elenco(perfis, por):
    medidas = [f"{nome}_recente" for nome in MEDIDAS]
    grupos_elenco = perfis.groupby(por, sort=False, dropna=False)
    medias = grupos_elenco[medidas].mean().where(grupos_elenco[medidas].count().ge(MINIMO_JOGADORES))
    medias["jogadores_com_forma"] = grupos_elenco["adr_recente"].count()
    medias["jogadores_com_tier"] = grupos_elenco["nivel_eventos_recente"].count()
    medias["saldo_dois_menores"] = grupos_elenco["saldo_mapa_recente"].agg(
        lambda valores: valores.nsmallest(2).mean() if valores.count() >= 4 else np.nan
    )
    return medias.reset_index()


consulta = jogadores[["match_id", "lado", "id", "dia"]].reset_index(drop=True)
consulta["consulta_id"] = consulta.index
forma_pre_jogo = consultar_jogadores(consulta)
elencos_pre_jogo = resumir_elenco(forma_pre_jogo, ["match_id", "lado"])
display(elencos_pre_jogo[[
    "match_id", "lado", "jogadores_com_forma", "adr_recente", "saldo_mapa_recente",
    "nivel_eventos_recente", "fracao_tier3_recente",
]].tail(10).round(3))

,match_id,lado,jogadores_com_forma,adr_recente,saldo_mapa_recente,nivel_eventos_recente,fracao_tier3_recente
22044,10097991,2,0,NaN,NaN,NaN,NaN
22045,7306089,2,0,NaN,NaN,NaN,NaN
22046,10098752,2,0,NaN,NaN,NaN,NaN
22047,10098754,2,0,NaN,NaN,NaN,NaN
22048,10098748,2,0,NaN,NaN,NaN,NaN
22049,10098749,2,0,NaN,NaN,NaN,NaN
22050,7310618,2,0,NaN,NaN,NaN,NaN
22051,7310616,2,0,NaN,NaN,NaN,NaN
22052,7310611,2,0,NaN,NaN,NaN,NaN
22053,7310605,2,0,NaN,NaN,NaN,NaN


### 6.4 Montar as entradas do confronto

Formamos as diferenças entre as escalações e juntamos as entradas da V3. Assim, o modelo mantém a força histórica da equipe e recebe também a forma dos jogadores que estão escalados e o contexto dos rivais que eles enfrentaram.

In [14]:
# Montar diferenças entre as escalações
ATRIBUTOS_FORMA = [f"{nome}_recente" for nome in METRICAS_INDIVIDUAIS] + ["saldo_dois_menores"]
ATRIBUTOS_DIFICULDADE = [f"{nome}_recente" for nome in CONTEXTO_INDIVIDUAL]
ENTRADAS_FORMA = [f"dif_{nome}" for nome in ATRIBUTOS_FORMA]
ENTRADAS_DIFICULDADE = [f"dif_{nome}" for nome in ATRIBUTOS_DIFICULDADE]
ENTRADAS_V4 = [*ENTRADAS, *ENTRADAS_FORMA, *ENTRADAS_DIFICULDADE]

dados_v4 = dados_v3.copy()
for lado in (1, 2):
    recorte = elencos_pre_jogo.loc[elencos_pre_jogo["lado"].eq(lado)].drop(columns="lado")
    recorte = recorte.rename(columns={coluna: f"team{lado}_{coluna}" for coluna in recorte if coluna != "match_id"})
    dados_v4 = dados_v4.merge(recorte, on="match_id", how="left", validate="one_to_one")
for atributo in ATRIBUTOS_FORMA + ATRIBUTOS_DIFICULDADE:
    dados_v4[f"dif_{atributo}"] = dados_v4[f"team1_{atributo}"] - dados_v4[f"team2_{atributo}"]

assert dados_v4["match_id"].equals(dados_v3["match_id"])
assert dados_v4[ENTRADAS].equals(dados_v3[ENTRADAS])
assert set(dados_v4[ENTRADAS_V4].columns).isdisjoint(["tier", ALVO])
display(dados_v4.groupby("tier").agg(
    series=("match_id", "size"),
    forma_dos_dois_lados=("dif_adr_recente", lambda s: s.notna().mean()),
    nivel_dos_dois_lados=("dif_nivel_eventos_recente", lambda s: s.notna().mean()),
).round(3))

,series,forma_dos_dois_lados,nivel_dos_dois_lados
tier,,,
Sem tier identificado,22,0.227,0.409
Tier 1,5190,0.753,0.840
Tier 2,5352,0.568,0.749
Tier 3,907,0.300,0.458


## 7. Treinar e medir a contribuição das novas entradas

Comparamos a V3, a V3 com desempenho individual, a V3 com dificuldade e a V4 com os dois blocos. Mantemos a regressão logística, a simetrização e os cortes cronológicos. A regressão aprende os pesos de Elo e desempenho individual.

O ajuste entre tiers favorece a escalação mais exposta a calendários fortes. Testamos três intensidades na validação, correspondentes a 52,5%, 55% e 57,5% de vitória para um Tier 1 contra um Tier 3 em um exemplo com os demais atributos iguais. Aplicamos a diferença em log-odds e usamos a menor perda logarítmica da validação para escolher a intensidade. Sem exposição suficiente em um dos lados, o ajuste vale zero. O período de teste é o mesmo já usado no projeto, para permitir comparação histórica.

In [15]:
# Regressão da V3 e ajuste de dificuldade que respeita o sentido da diferença de tiers
from scipy.special import expit, logit

COLUNAS_TIERS = ["dif_nivel_eventos_recente", "dif_fracao_tier3_recente"]
COLUNAS_ELO = [coluna for coluna in ENTRADAS_DIFICULDADE if coluna not in COLUNAS_TIERS]
ENTRADAS_PIPELINE_V4 = [coluna for coluna in ENTRADAS_V4 if coluna not in COLUNAS_TIERS]


class ModeloComDificuldade:
    def __init__(self, modelo, entradas_pipeline, multiplicador):
        self.modelo = modelo
        self.entradas_pipeline = entradas_pipeline
        self.multiplicador = multiplicador
        self.classes_ = modelo.classes_

    def predict_proba(self, x):
        # Tier 1 puro tem índice 1, Tier 2 puro tem 0,75 e Tier 3 puro tem 0.
        diferenca_nivel = (
            0.25 * x["dif_nivel_eventos_recente"] - 0.5 * x["dif_fracao_tier3_recente"]
        ).fillna(0.0)
        base = self.modelo.predict_proba(x[self.entradas_pipeline])[:, 1]
        ajustada = expit(logit(base) + self.multiplicador * diferenca_nivel.to_numpy())
        return np.column_stack([1.0 - ajustada, ajustada])


def treinar_pipeline(colunas):
    treino = dados_v4.loc[particoes["Treino"]]
    return reconstrucao.new_pipeline().fit(treino[colunas], treino[ALVO])


pipeline_v4 = treinar_pipeline(ENTRADAS_PIPELINE_V4)
simetrico_v4 = ModeloSimetrico(pipeline_v4)
val = dados_v4.loc[particoes["Validação"]]
calibracao = []
for vantagem in (0.525, 0.55, 0.575):
    candidato = ModeloComDificuldade(simetrico_v4, ENTRADAS_PIPELINE_V4, float(logit(vantagem)))
    predicao = candidato.predict_proba(val[ENTRADAS_V4])[:, 1]
    calibracao.append({
        "Tier 1 x Tier 3 com resto igual (%)": 100 * vantagem,
        "ajuste máximo em log-odds": float(logit(vantagem)),
        "Log Loss de validação": log_loss(val[ALVO], predicao, labels=[0, 1]),
    })
calibracao = pd.DataFrame(calibracao).sort_values("Log Loss de validação").reset_index(drop=True)
ajuste_tier = float(calibracao.loc[0, "ajuste máximo em log-odds"])
modelo_v4 = ModeloComDificuldade(simetrico_v4, ENTRADAS_PIPELINE_V4, ajuste_tier)
pipeline_dificuldade = ENTRADAS + COLUNAS_ELO
modelo_dificuldade = ModeloComDificuldade(
    ModeloSimetrico(treinar_pipeline(pipeline_dificuldade)), pipeline_dificuldade, ajuste_tier,
)
entradas_so_forma = ENTRADAS + ENTRADAS_FORMA
variantes = {
    "V3": (modelo_v3, ENTRADAS),
    "V3 + forma individual": (ModeloSimetrico(treinar_pipeline(entradas_so_forma)), entradas_so_forma),
    "V3 + dificuldade": (modelo_dificuldade, pipeline_dificuldade + COLUNAS_TIERS),
    "V4": (modelo_v4, ENTRADAS_V4),
}

amostra = dados_v4.loc[particoes["Validação"], ENTRADAS_V4].head(64)
invertida = amostra.copy()
colunas_dif = [coluna for coluna in ENTRADAS_V4 if coluna.startswith("dif_")]
invertida[colunas_dif] = -invertida[colunas_dif]
np.testing.assert_allclose(
    modelo_v4.predict_proba(amostra)[:, 1] + modelo_v4.predict_proba(invertida)[:, 1],
    1.0, atol=1e-12,
)
print(f"Treino de {particoes['Treino'].sum():,} séries · V3 com {len(ENTRADAS)} entradas · V4 com {len(ENTRADAS_V4)}.")
assert np.isfinite(pipeline_v4.named_steps["imputacao"].statistics_).all()
display(calibracao.round(6))
display(pd.DataFrame({
    "entrada do pipeline": ENTRADAS_PIPELINE_V4,
    "coeficiente padronizado": pipeline_v4.named_steps["classificador"].coef_[0][:len(ENTRADAS_PIPELINE_V4)],
}).round(4))

# Isolar o efeito aprendido dos tiers em um confronto ilustrativo, com o restante igual.
exemplo_tiers = pd.DataFrame(0.0, index=["mesmo nível", "Tier 1 contra Tier 3", "Tier 3 contra Tier 1"],
                             columns=ENTRADAS_V4)
exemplo_tiers["bestOf"] = 3
exemplo_tiers["dif_nivel_eventos_recente"] = [0.0, 2.0, -2.0]
exemplo_tiers["dif_fracao_tier3_recente"] = [0.0, -1.0, 1.0]
efeito_tiers = pd.DataFrame({
    "chance de A vencer (%)": 100 * modelo_v4.predict_proba(exemplo_tiers)[:, 1],
}, index=exemplo_tiers.index)
assert efeito_tiers.iloc[1, 0] > 50 > efeito_tiers.iloc[2, 0]
np.testing.assert_allclose(efeito_tiers.iloc[1:, 0].sum(), 100.0)
display(efeito_tiers.round(2))

Treino de 8,109 séries · V3 com 5 entradas · V4 com 13.


,Tier 1 x Tier 3 com resto igual (%),ajuste máximo em log-odds,Log Loss de validação
0,52.5,0.100083,0.653517
1,55.0,0.200671,0.653922
2,57.5,0.302281,0.654365


,entrada do pipeline,coeficiente padronizado
0,dif_jogos_anteriores,0.3795
1,dif_vitorias_anteriores,-0.2448
2,dif_taxa_vitorias_anteriores,0.1781
3,bestOf,-0.0131
4,dif_pontos_hltv,0.1837
5,dif_adr_recente,0.0986
6,dif_kast_recente,0.0127
7,dif_saldo_mapa_recente,0.0915
8,dif_saldo_dois_menores,-0.0234
9,dif_elo_equipe_recente,0.0728


,chance de A vencer (%)
mesmo nível,50.0
Tier 1 contra Tier 3,52.5
Tier 3 contra Tier 1,47.5


### 7.1 Validar em séries futuras e em confrontos com desnível

O recorte de desnível considera uma equipe com metade ou mais da exposição recente em Tier 3 e a outra com maioria em Tier 1 e 2. O cálculo vem das escalações antes de cada série. A perda logarítmica e o Brier avaliam as probabilidades, a acurácia conta os vencedores apontados e a AUC mede a ordenação.

In [16]:
# Comparação em validação e teste, com os mesmos confrontos nas quatro versões
LIMITE_POUCO_HISTORICO = 10
GRUPO_POUCO_HISTORICO = "Com equipe abaixo de 10 séries anteriores no Tier 1"
GRUPO_DESNIVEL = "Escalação mais exposta ao Tier 3 contra escalação de Tier 1/2"
colunas_volume = [f"team{lado}_jogos_anteriores" for lado in (1, 2)]
pouco_historico = dados_ref[colunas_volume].lt(LIMITE_POUCO_HISTORICO)
tiers_jogados = dados_v4[["team1_fracao_tier3_recente", "team2_fracao_tier3_recente"]]
predominio_tier3 = tiers_jogados.ge(0.5)
desnivel = tiers_jogados.notna().all(axis=1) & predominio_tier3.sum(axis=1).eq(1)
grupos = {
    "Todas as séries": pd.Series(True, index=dados_v4.index),
    "Apenas Tier 1": dados_v4["tier"].eq("Tier 1"),
    "Tier 2 e 3": dados_v4["tier"].isin(["Tier 2", "Tier 3"]),
    GRUPO_POUCO_HISTORICO: pouco_historico.any(axis=1),
    GRUPO_DESNIVEL: desnivel,
}


def calcular_metricas(alvo, probabilidade):
    return {
        "Log Loss": log_loss(alvo, probabilidade, labels=[0, 1]),
        "Brier": brier_score_loss(alvo, probabilidade),
        "Acurácia": accuracy_score(alvo, (probabilidade > 0.5).astype(int)),
        "AUC": roc_auc_score(alvo, probabilidade) if alvo.nunique() == 2 else np.nan,
    }


def avaliar(conjunto):
    selecao = particoes[conjunto]
    alvo = dados_v4.loc[selecao, ALVO]
    probabilidades = {
        "Referência HLTV": modelo_ref.predict_proba(dados_ref.loc[selecao, ENTRADAS])[:, 1],
        **{
            nome: estimador.predict_proba(dados_v4.loc[selecao, colunas])[:, 1]
            for nome, (estimador, colunas) in variantes.items()
        },
        "50%": 0.5,
    }
    probs = pd.DataFrame(probabilidades, index=alvo.index)
    resultados = []
    for nome, pertence in grupos.items():
        ids = alvo.index[pertence.loc[alvo.index]]
        if len(ids):
            for versao in probs:
                resultados.append({
                    "grupo": nome, "modelo": versao, "séries": len(ids),
                    **calcular_metricas(alvo.loc[ids], probs.loc[ids, versao]),
                })
    return pd.DataFrame(resultados).set_index(["grupo", "modelo"]), probs


metricas_validacao, prob_validacao = avaliar("Validação")
metricas_teste, prob_teste = avaliar("Teste")
print("Validação · janeiro a junho de 2026")
display(metricas_validacao.round(4))
print("Teste · julho a novembro de 2026")
display(metricas_teste.round(4))

Validação · janeiro a junho de 2026
Teste · julho a novembro de 2026


séries  ...     AUC
grupo                                              modelo                         ...        
Todas as séries                                    Referência HLTV          1844  ...  0.6100
                                                   V3                       1844  ...  0.6327
                                                   V3 + forma individual    1844  ...  0.6402
                                                   V3 + dificuldade         1844  ...  0.6321
                                                   V4                       1844  ...  0.6382
                                                   50%                      1844  ...  0.5000
Apenas Tier 1                                      Referência HLTV           642  ...  0.6433
                                                   V3                        642  ...  0.6379
                                                   V3 + forma individual     642  ...  0.6499
                                                   V3 + dificuldade          642  ...  0.6599
                                                   V4                        642  ...  0.6700
                                                   50%                       642  ...  0.5000
Tier 2 e 3                                         Referência HLTV          1202  ...  0.5926
                                                   V3                       1202  ...  0.6337
                                                   V3 + forma individual    1202  ...  0.6402
                                                   V3 + dificuldade         1202  ...  0.6186
                                                   V4                       1202  ...  0.6242
                                                   50%                      1202  ...  0.5000
Com equipe abaixo de 10 séries anteriores no Ti... Referência HLTV          1121  ...  0.6103
                                                   V3                       1121  ...  0.6315
                                                   V3 + forma individual    1121  ...  0.6337
                                                   V3 + dificuldade         1121  ...  0.6285
                                                   V4                       1121  ...  0.6294
                                                   50%                      1121  ...  0.5000
Escalação mais exposta ao Tier 3 contra escalaç... Referência HLTV            44  ...  0.6253
                                                   V3                         44  ...  0.6358
                                                   V3 + forma individual      44  ...  0.6021
                                                   V3 + dificuldade           44  ...  0.7053
                                                   V4                         44  ...  0.6779
                                                   50%                        44  ...  0.5000

[30 rows x 5 columns]

séries  ...     AUC
grupo                                              modelo                         ...        
Todas as séries                                    Referência HLTV          1518  ...  0.6389
                                                   V3                       1518  ...  0.6460
                                                   V3 + forma individual    1518  ...  0.6481
                                                   V3 + dificuldade         1518  ...  0.6444
                                                   V4                       1518  ...  0.6473
                                                   50%                      1518  ...  0.5000
Apenas Tier 1                                      Referência HLTV           773  ...  0.6350
                                                   V3                        773  ...  0.6520
                                                   V3 + forma individual     773  ...  0.6573
                                                   V3 + dificuldade          773  ...  0.6390
                                                   V4                        773  ...  0.6414
                                                   50%                       773  ...  0.5000
Tier 2 e 3                                         Referência HLTV           723  ...  0.6492
                                                   V3                        723  ...  0.6473
                                                   V3 + forma individual     723  ...  0.6474
                                                   V3 + dificuldade          723  ...  0.6575
                                                   V4                        723  ...  0.6603
                                                   50%                       723  ...  0.5000
Com equipe abaixo de 10 séries anteriores no Ti... Referência HLTV           864  ...  0.6607
                                                   V3                        864  ...  0.6620
                                                   V3 + forma individual     864  ...  0.6659
                                                   V3 + dificuldade          864  ...  0.6636
                                                   V4                        864  ...  0.6674
                                                   50%                       864  ...  0.5000
Escalação mais exposta ao Tier 3 contra escalaç... Referência HLTV            20  ...  0.6615
                                                   V3                         20  ...  0.6354
                                                   V3 + forma individual      20  ...  0.6458
                                                   V3 + dificuldade           20  ...  0.6875
                                                   V4                         20  ...  0.6354
                                                   50%                        20  ...  0.5000

[30 rows x 5 columns]

## 8. Auditar mapas, medir força e prever o veto

### 8.1 Extrair mapas jogados e sequências completas

Ajustamos a orientação dos lados pelo jogo de origem e usamos os placares registrados. O Elo aprende com os mapas válidos. Para as preferências de veto, exigimos que a sequência ordenada por ID reproduza o placar e o encerramento da série. Um mapa decisivo ausente em uma vitória por 2–0 fica sem observação nessa posição. O CSV registra os mapas disputados, portanto estimamos a **saída** do veto, com as ações de ban e pick fora da base.

In [17]:
# Mapas com placar verificável e ordem compatível com o encerramento da série
from bisect import bisect_left
from collections import Counter, defaultdict
from functools import lru_cache
from itertools import permutations

MAPAS = ("Ancient", "Anubis", "Cache", "Dust2", "Inferno", "Mirage", "Nuke", "Overpass", "Train", "Vertigo")
MAPAS_MAJOR = MAPAS[:7]
FORMATOS = (1, 3, 5)
mapas_brutos = pd.read_csv(caminho_csv, usecols=[
    "match_id", "game_id", "is_total", "team1", "team2", "map_name", "score1_game", "score2_game",
])
linhas_mapas = mapas_brutos.loc[
    mapas_brutos["is_total"].eq(False) & mapas_brutos["map_name"].isin(MAPAS)
    & mapas_brutos["game_id"].gt(0)
    & mapas_brutos[["score1_game", "score2_game"]].notna().all(axis=1)
    & mapas_brutos["score1_game"].ne(mapas_brutos["score2_game"])
].rename(columns={"team1": "time_a_mapa", "team2": "time_b_mapa"})
dados_series = series_com_jogadores[[
    "match_id", "datetime", "team1", "team2", "bestOf", "games_played", "score1_match", "score2_match",
]]
mapas = linhas_mapas.merge(dados_series, on="match_id", how="inner", validate="many_to_one")
a = mapas["time_a_mapa"].str.casefold()
b = mapas["time_b_mapa"].str.casefold()
t1 = mapas["team1"].str.casefold()
t2 = mapas["team2"].str.casefold()
em_ordem = a.eq(t1) & b.eq(t2)
invertidos = a.eq(t2) & b.eq(t1)
mapas["venceu_mapa_a"] = np.where(
    em_ordem, mapas["score1_game"].gt(mapas["score2_game"]), mapas["score2_game"].gt(mapas["score1_game"]),
)
mapas = mapas.loc[em_ordem | invertidos].copy()
n_duplicados = mapas.duplicated(["match_id", "map_name"], keep=False)
mapas = mapas.loc[~n_duplicados].sort_values("game_id").reset_index(drop=True)
mapas["dia"] = mapas["datetime"].dt.normalize()
mapas["venceu_mapa_b"] = ~mapas["venceu_mapa_a"]
mapas["posicao"] = mapas.groupby("match_id").cumcount() + 1
auditoria_mapas = mapas.groupby("match_id").agg(
    registros=("game_id", "size"), vitorias_a=("venceu_mapa_a", "sum"),
    vitorias_b=("venceu_mapa_b", "sum"), ultimo_para_a=("venceu_mapa_a", "last"),
).join(dados_series.set_index("match_id"))
vence_a_serie = auditoria_mapas["score1_match"].gt(auditoria_mapas["score2_match"])
serie_consistente = (
    auditoria_mapas["bestOf"].isin(FORMATOS)
    & auditoria_mapas["registros"].eq(auditoria_mapas["games_played"])
    & auditoria_mapas["registros"].le(auditoria_mapas["bestOf"])
    & auditoria_mapas["vitorias_a"].eq(auditoria_mapas["score1_match"])
    & auditoria_mapas["vitorias_b"].eq(auditoria_mapas["score2_match"])
    & auditoria_mapas["ultimo_para_a"].eq(vence_a_serie)
    & auditoria_mapas[["vitorias_a", "vitorias_b"]].max(axis=1).eq(auditoria_mapas["bestOf"] // 2 + 1)
)
ids_veto = auditoria_mapas.index[serie_consistente]
mapas_veto = mapas.loc[mapas["match_id"].isin(ids_veto)].copy()
assert mapas_veto["posicao"].le(mapas_veto["bestOf"]).all()
print(f"{len(mapas):,} mapas com placar alinhado para Elo · {len(ids_veto):,} séries completas para o veto.")
display(auditoria_mapas.loc[serie_consistente].groupby("bestOf").agg(
    series=("registros", "size"), mapas_jogados=("registros", "sum"),
))
display(mapas["map_name"].value_counts().rename("mapas auditados").to_frame())

12,524 mapas com placar alinhado para Elo · 4,974 séries completas para o veto.


,series,mapas_jogados
bestOf,,
1.0,799,799
3.0,4109,9906
5.0,66,251


,mapas auditados
map_name,
Ancient,2164
Mirage,2155
Nuke,1803
Dust2,1667
Inferno,1579
Anubis,1409
Overpass,614
Vertigo,549
Train,333


### 8.2 Calcular Elo antes de cada dia e usar os mapas elegíveis

O Elo por mapa parte de 1500, com atualização de 24 pontos multiplicados pelo erro de previsão. A cada 180 dias, a distância até a nota inicial cai pela metade. Todas as consultas usam estados anteriores ao dia do jogo. A falta de jogos naquele mapa usa a nota inicial.

Usamos uma aproximação dos pools históricos com transições para torneios que aplicaram as mudanças em datas diferentes. O pool em Singapura tem **Ancient, Anubis, Cache, Dust II, Inferno, Mirage e Nuke**.

[Mapas de Singapura](https://www.hltv.org/events/8302/pgl-major-singapore-2026) · [Dust II](https://store.steampowered.com/news/app/730/view/4202497395513061959) · [Train](https://store.steampowered.com/news/posts/?appgroupname=Counter-Strike%3A+Global+Offensive&appids=730&enddate=1738630258&feed=steam_community_announcements) · [Overpass](https://steamcommunity.com/games/CSGO/announcements/detail/529852487375519750) · [Anubis](https://store.steampowered.com/news/posts/?appids=730&enddate=1770248032&feed=steam_community_announcements) · [Cache](https://store.steampowered.com/news/posts/?appids=730&enddate=1787182592)

In [18]:
# Pools históricos aproximados e Elo do mapa, com atualizações ao fim do dia
ORIGEM_MAPAS = pd.Timestamp("2023-01-01")
MEIA_VIDA_MAPAS = 180
K_ELO_MAPA = 24.0
ESCALA_ELO = np.log(10.0) / 400.0
POOL_INICIAL = {"Ancient", "Anubis", "Inferno", "Mirage", "Nuke", "Overpass", "Vertigo"}
TRANSICOES = [
    ("2024-04-25", "2024-05-09", "Overpass", "Dust2"),
    ("2025-01-06", "2025-02-12", "Vertigo", "Train"),
    ("2025-07-15", "2025-07-29", "Anubis", "Overpass"),
    ("2026-01-21", "2026-02-05", "Train", "Anubis"),
    ("2026-06-22", "2026-07-22", "Overpass", "Cache"),
]
TRANSICOES = [
    (pd.Timestamp(inicio), pd.Timestamp(fim), sai, entra) for inicio, fim, sai, entra in TRANSICOES
]


def pool_historico(dia):
    pool = POOL_INICIAL.copy()
    for inicio, fim, sai, entra in TRANSICOES:
        if dia < inicio:
            break
        pool.add(entra)
        if dia <= fim:
            break
        pool.remove(sai)
    return tuple(mapa for mapa in MAPAS if mapa in pool)


def dia_num(dia):
    return (dia - ORIGEM_MAPAS).days


def decaimento(dias):
    return np.exp2(-dias / MEIA_VIDA_MAPAS)


def construir_elo_mapas():
    ultimo = {}
    historico = defaultdict(lambda: ([], [], []))
    for data, grupo in mapas.groupby("dia", sort=True):
        hoje = dia_num(data)
        incremento = defaultdict(float)
        contagem = Counter()
        for time_a, time_b, mapa, venceu in grupo[
            ["team1", "team2", "map_name", "venceu_mapa_a"]
        ].itertuples(index=False, name=None):
            chave_a = (time_a, mapa)
            chave_b = (time_b, mapa)
            dia_a, elo_a, _ = ultimo.get(chave_a, (hoje, 0.0, 0))
            dia_b, elo_b, _ = ultimo.get(chave_b, (hoje, 0.0, 0))
            diferenca = elo_a * decaimento(hoje - dia_a) - elo_b * decaimento(hoje - dia_b)
            ajuste = K_ELO_MAPA * (venceu - expit(diferenca * ESCALA_ELO))
            incremento[chave_a] += ajuste
            incremento[chave_b] -= ajuste
            contagem.update((chave_a, chave_b))
        for chave, ajuste in incremento.items():
            dia_ant, elo_ant, n_ant = ultimo.get(chave, (hoje, 0.0, 0))
            elo_novo = elo_ant * decaimento(hoje - dia_ant) + ajuste
            n_novo = n_ant + contagem[chave]
            ultimo[chave] = (hoje, elo_novo, n_novo)
            dias, notas, volumes = historico[chave]
            dias.append(hoje)
            notas.append(elo_novo)
            volumes.append(n_novo)
    return dict(historico)


elo_mapas = construir_elo_mapas()


@lru_cache(maxsize=30000)
def consultar_elo_mapa(equipe, mapa, hoje):
    estado = elo_mapas.get((equipe, mapa))
    if estado is None:
        return 0.0, 0
    dias, notas, volumes = estado
    i = bisect_left(dias, hoje) - 1
    if i < 0:
        return 0.0, 0
    return notas[i] * decaimento(hoje - dias[i]), volumes[i]


assert pool_historico(pd.Timestamp("2026-11-17")) == MAPAS_MAJOR
assert all(consultar_elo_mapa(equipe, mapa, data[0]) == (0.0, 0)
           for (equipe, mapa), (data, _, _) in list(elo_mapas.items())[:100])

### 8.3 Integrar probabilidades sobre as possíveis sequências

Estimamos a chance de cada mapa em cada posição usando ocorrências anteriores, com peso maior para as recentes. Quando há pouco histórico naquele formato e posição, usamos a frequência geral da equipe e da base. A média geométrica dos perfis dos dois times dá o peso do mapa para aquela partida.

Em MD1, a chance condicionada ao veto é a média das probabilidades de vitória ponderada pelos possíveis mapas sobreviventes. Em MD3 e MD5, enumeramos sequências sem repetição. Para cada sequência calculamos a chance de vencer a maioria dos mapas, tratando os resultados de cada mapa como independentes dada a sequência. O total pondera cada sequência pela probabilidade de ela sair.

In [19]:
# Distribuição das posições observadas, condicionada ao formato e às equipes
indice_mapa = {nome: i for i, nome in enumerate(MAPAS)}
GLOBAL = "__todas_as_equipes__"


def perspectivas_mapa(base, posicional):
    partes = []
    for lado in (1, 2):
        dados = base[[f"team{lado}", "dia", "map_name", "bestOf", "posicao"]].rename(
            columns={f"team{lado}": "equipe"},
        )
        if not posicional:
            dados = dados.assign(bestOf=0, posicao=0)
        partes.append(dados)
    return pd.concat(partes, ignore_index=True)


observacoes = pd.concat([
    perspectivas_mapa(mapas, False),
    perspectivas_mapa(mapas_veto, True),
], ignore_index=True)
observacoes = pd.concat([observacoes, observacoes.assign(equipe=GLOBAL)], ignore_index=True)
observacoes["dia_num"] = (observacoes["dia"] - ORIGEM_MAPAS).dt.days
blocos_veto = (
    observacoes.groupby(["equipe", "bestOf", "posicao", "dia_num", "map_name"]).size()
    .unstack("map_name", fill_value=0).reindex(columns=MAPAS, fill_value=0)
    .reset_index().sort_values("dia_num")
)
historico_veto = {}
for chave, grupo in blocos_veto.groupby(["equipe", "bestOf", "posicao"], sort=False):
    dias = grupo["dia_num"].to_numpy()
    pesos = np.exp2(dias / MEIA_VIDA_MAPAS)
    historico_veto[chave] = (dias, (grupo[list(MAPAS)].to_numpy() * pesos[:, None]).cumsum(axis=0))


@lru_cache(maxsize=30000)
def ocorrencias(equipe, formato, posicao, hoje):
    estado = historico_veto.get((equipe, formato, posicao))
    if estado is not None:
        dias, somas = estado
        i = bisect_left(dias, hoje) - 1
        if i >= 0:
            return somas[i] * decaimento(hoje)
    return np.zeros(len(MAPAS))


def preferencia(equipe, formato, posicao, hoje, indices, geral, referencia_posicao):
    contagem_equipe = ocorrencias(equipe, 0, 0, hoje)[indices] + 12 * geral
    perfil_equipe = contagem_equipe / contagem_equipe.sum()
    priori = 0.65 * perfil_equipe + 0.35 * referencia_posicao
    contagem_posicao = ocorrencias(equipe, formato, posicao, hoje)[indices] + 8 * priori
    return contagem_posicao / contagem_posicao.sum()


def pesos_de_veto(equipe_a, equipe_b, formato, hoje, pool):
    indices = [indice_mapa[nome] for nome in pool]
    ocorrencia_geral = ocorrencias(GLOBAL, 0, 0, hoje)[indices] + 1
    geral = ocorrencia_geral / ocorrencia_geral.sum()
    probabilidades = []
    for posicao in range(1, formato + 1):
        obs = ocorrencias(GLOBAL, formato, posicao, hoje)[indices] + 12 * geral
        referencia_posicao = obs / obs.sum()
        perfil_a = preferencia(equipe_a, formato, posicao, hoje, indices, geral, referencia_posicao)
        perfil_b = preferencia(equipe_b, formato, posicao, hoje, indices, geral, referencia_posicao)
        pesos = np.sqrt(perfil_a * perfil_b)
        probabilidades.append(pesos / pesos.sum())
    return np.asarray(probabilidades)


@lru_cache(maxsize=12)
def sequencias_possiveis(tamanho_pool, formato):
    return np.asarray(list(permutations(range(tamanho_pool), formato)), dtype=np.int8)


def integrar_mapas(equipe_a, equipe_b, formato, hoje, pool):
    pesos = pesos_de_veto(equipe_a, equipe_b, formato, hoje, pool)
    p_mapa = expit(ESCALA_ELO * np.array([
        consultar_elo_mapa(equipe_a, mapa, hoje)[0] - consultar_elo_mapa(equipe_b, mapa, hoje)[0]
        for mapa in pool
    ]))
    sequencias = sequencias_possiveis(len(pool), formato)
    peso_sequencia = np.ones(len(sequencias))
    resultados = np.ones((len(sequencias), 1))
    for i, linha in enumerate(pesos):
        restante = 1 - linha[sequencias[:, :i]].sum(axis=1)
        peso_sequencia *= linha[sequencias[:, i]] / restante
        p = p_mapa[sequencias[:, i], None]
        resultados = np.pad(resultados * (1 - p), ((0, 0), (0, 1))) + np.pad(
            resultados * p, ((0, 0), (1, 0)),
        )
    vitoria = resultados[:, formato // 2 + 1:].sum(axis=1)
    assert np.isclose(peso_sequencia.sum(), 1, atol=1e-10)
    return float(peso_sequencia @ vitoria), float(vitoria.mean()), pesos[0], p_mapa


a, b = "Team Liquid", "Team Falcons"
for formato in FORMATOS:
    hoje = dia_num(pd.Timestamp("2026-11-17"))
    ida = integrar_mapas(a, b, formato, hoje, MAPAS_MAJOR)
    volta = integrar_mapas(b, a, formato, hoje, MAPAS_MAJOR)
    np.testing.assert_allclose([ida[0] + volta[0], ida[1] + volta[1]], [1, 1])
    np.testing.assert_allclose(ida[2], volta[2])
    if formato == 1:
        np.testing.assert_allclose(ida[0], ida[2] @ ida[3])

## 9. Treinar e testar a V5

As entradas novas são a vantagem prevista na série com mapas ponderados pelo veto e a diferença entre essa previsão e um sorteio uniforme de mapas. A regressão recebe todas as entradas da V4 junto dessas duas. Aplicamos a mesma correção de tiers da V4.

In [20]:
# Usar as probabilidades de mapas como duas novas entradas da regressão
COLUNAS_MAPAS = ["dif_vitoria_mapas_provaveis", "dif_efeito_veto"]
ENTRADAS_V5 = [*ENTRADAS_V4, *COLUNAS_MAPAS]
ENTRADAS_PIPELINE_V5 = [*ENTRADAS_PIPELINE_V4, *COLUNAS_MAPAS]
calendario = {dia: pool_historico(dia) for dia in mapas["dia"].unique()}

variaveis = []
primeiro_mapa = {}
for confronto, data, time_a, time_b, formato in dados_v4[
    ["match_id", "datetime", "team1", "team2", "bestOf"]
].itertuples(index=False, name=None):
    if formato not in FORMATOS:
        continue
    dia = data.normalize()
    pool = calendario.get(dia, pool_historico(dia))
    com_veto, uniforme, distribuicao, _ = integrar_mapas(time_a, time_b, int(formato), dia_num(dia), pool)
    # Ao inverter as equipes, as duas diferenças trocam de sinal.
    variaveis.append((confronto, 2.0 * com_veto - 1.0, com_veto - uniforme))
    primeiro_mapa[confronto] = dict(zip(pool, distribuicao, strict=True))
mapas_pre_jogo = pd.DataFrame(variaveis, columns=["match_id", *COLUNAS_MAPAS])
dados_v5 = dados_v4.merge(mapas_pre_jogo, on="match_id", how="left", validate="one_to_one")
assert dados_v5[["match_id", *ENTRADAS_V4]].equals(dados_v4[["match_id", *ENTRADAS_V4]])
assert len(ENTRADAS_V5) == len(ENTRADAS_V4) + 2

treino = particoes["Treino"]
pipeline_v5 = reconstrucao.new_pipeline().fit(
    dados_v5.loc[treino, ENTRADAS_PIPELINE_V5], dados_v5.loc[treino, ALVO],
)
modelo_v5 = ModeloComDificuldade(
    ModeloSimetrico(pipeline_v5), ENTRADAS_PIPELINE_V5, ajuste_tier,
)
print(f"{len(ENTRADAS_V4)} entradas da V4 + {len(COLUNAS_MAPAS)} de mapas = {len(ENTRADAS_V5)} entradas na V5.")
display(dados_v5.groupby("tier").agg(
    series=("match_id", "size"), formatos_com_mapas=("dif_vitoria_mapas_provaveis", "count"),
))

13 entradas da V4 + 2 de mapas = 15 entradas na V5.


,series,formatos_com_mapas
tier,,
Sem tier identificado,22,22
Tier 1,5190,5110
Tier 2,5352,5313
Tier 3,907,906


### 9.1 Comparar no mesmo teste cronológico

Medimos a previsão do primeiro mapa nos confrontos com sequência validada e o acerto de Elo dado o mapa efetivamente jogado. Na linha de perda do veto usamos os mapas que pertencem ao pool previsto e mostramos a cobertura. Para prever a série inteira, V4 e V5 são avaliadas nos **mesmos confrontos**, com as probabilidades de mapa construídas antes de cada jogo.

In [21]:
# Testar o modelo de mapas e a previsão de séries em períodos posteriores ao treino
registros_veto = []
for confronto, mapa, formato, data in mapas_veto.loc[
    mapas_veto["posicao"].eq(1) & mapas_veto["dia"].ge(INICIO_VALIDACAO),
    ["match_id", "map_name", "bestOf", "dia"],
].itertuples(index=False, name=None):
    pesos = primeiro_mapa[confronto]
    observado_no_pool = mapa in pesos
    registros_veto.append({
        "janela": "Teste" if data >= INICIO_TESTE else "Validação",
        "formato": f"MD{int(formato)}",
        "no_pool": observado_no_pool,
        "mapa_top1": max(pesos, key=pesos.get) == mapa,
        "perda_veto": -np.log(pesos[mapa]) if observado_no_pool else np.nan,
        "perda_uniforme": np.log(len(pesos)) if observado_no_pool else np.nan,
    })
avaliacao_veto = pd.DataFrame(registros_veto).groupby(["janela", "formato"]).agg(
    series_auditadas=("no_pool", "size"),
    mapas_no_pool=("no_pool", "sum"),
    acerto_primeiro_mapa=("mapa_top1", "mean"),
    perda_log_modelo=("perda_veto", "mean"),
    perda_log_uniforme=("perda_uniforme", "mean"),
)
display(avaliacao_veto.round(4))

amostra_mapas = mapas.loc[mapas["dia"].ge(INICIO_TESTE)].copy()
prob_elo_mapa = [
    expit(ESCALA_ELO * (
        consultar_elo_mapa(a, mapa, dia_num(dia))[0] - consultar_elo_mapa(b, mapa, dia_num(dia))[0]
    ))
    for a, b, mapa, dia in amostra_mapas[["team1", "team2", "map_name", "dia"]].itertuples(index=False, name=None)
]
print("Elo de mapas no teste com o mapa jogado informado")
display(pd.DataFrame([
    {"modelo": "Elo por mapa", "mapas": len(amostra_mapas),
     **calcular_metricas(amostra_mapas["venceu_mapa_a"], np.asarray(prob_elo_mapa))},
    {"modelo": "50% por mapa", "mapas": len(amostra_mapas),
     **calcular_metricas(amostra_mapas["venceu_mapa_a"], np.full(len(amostra_mapas), 0.5))},
]).set_index("modelo").round(4))

historico_modelos = []
for janela in ("Validação", "Teste"):
    recorte = dados_v5.loc[particoes[janela]]
    valores = {
        "V4": modelo_v4.predict_proba(recorte[ENTRADAS_V4])[:, 1],
        "V5": modelo_v5.predict_proba(recorte[ENTRADAS_V5])[:, 1],
    }
    for formato, filtro in {
        "Todas": np.ones(len(recorte), dtype=bool),
        "MD1": recorte["bestOf"].eq(1).to_numpy(),
        "MD3": recorte["bestOf"].eq(3).to_numpy(),
        "MD5": recorte["bestOf"].eq(5).to_numpy(),
    }.items():
        for versao, prob in valores.items():
            historico_modelos.append({
                "janela": janela, "formato": formato, "modelo": versao, "séries": int(filtro.sum()),
                **calcular_metricas(recorte[ALVO].loc[filtro], prob[filtro]),
            })
metricas_v5 = pd.DataFrame(historico_modelos).set_index(["janela", "formato", "modelo"])
display(metricas_v5.round(4))

Elo de mapas no teste com o mapa jogado informado


series_auditadas  ...  perda_log_uniforme
janela    formato                    ...                    
Teste     MD1                    97  ...              1.9652
          MD3                   568  ...              1.9520
          MD5                    13  ...              1.9665
Validação MD1                    88  ...              1.9459
          MD3                   609  ...              1.9669
          MD5                    14  ...              1.9650

[6 rows x 5 columns]

,mapas,Log Loss,Brier,Acurácia,AUC
modelo,,,,,
Elo por mapa,1829,0.6837,0.2453,0.5560,0.5765
50% por mapa,1829,0.6931,0.2500,0.4718,0.5000


séries  Log Loss   Brier  Acurácia     AUC
janela    formato modelo                                            
Validação Todas   V4        1844    0.6535  0.2311    0.6182  0.6382
                  V5        1844    0.6544  0.2314    0.6133  0.6374
          MD1     V4         179    0.6275  0.2192    0.6369  0.6318
                  V5         179    0.6259  0.2183    0.6313  0.6377
          MD3     V4        1638    0.6556  0.2320    0.6172  0.6387
                  V5        1638    0.6568  0.2324    0.6123  0.6372
          MD5     V4          21    0.6798  0.2443    0.5238  0.6091
                  V5          21    0.6818  0.2454    0.5238  0.5636
Teste     Todas   V4        1518    0.6570  0.2315    0.6021  0.6473
                  V5        1518    0.6578  0.2318    0.6080  0.6475
          MD1     V4         182    0.6360  0.2220    0.6319  0.6590
                  V5         182    0.6336  0.2212    0.6319  0.6616
          MD3     V4        1304    0.6619  0.2337    0.5959  0.6408
                  V5        1304    0.6630  0.2341    0.6028  0.6406
          MD5     V4          28    0.5843  0.2000    0.6786  0.7590
                  V5          28    0.5912  0.2029    0.7143  0.7590

## 10. Projetar o Major de Singapura

Usamos os resultados locais até **14/11**, o ranking de **16/11** e o corte de **17/11/2026**. As escalações de jogo incluem history na 9z, SunPayus na FUT, Anlelele na Astralis, apEX na Vitality e ksloks na Legacy, conforme os anúncios anteriores ao corte. O chaveamento segue as regras arquivadas da PGL. [Regras e etapas da PGL](https://www.pglesports.com/cs2/pgl-major-singapore-2026/) · [Escalações divulgadas](https://www.hltv.org/news/45950/pgl-reveals-singapore-major-rosters)

In [22]:
pasta_major = PASTA_DADOS / "singapura_2026"
ponteiro = json.loads((pasta_major / "ultima_preparacao.json").read_bytes())
fontes_major = pasta_major / PureWindowsPath(ponteiro["pasta_fontes"]).name
configuracao = json.loads((fontes_major / "config_torneio.json").read_bytes())

with patch.object(simulador, "PROJECT", PASTA_DADOS):
    with patch.object(simulador, "base_v1", return_value=(jogos, series_v3)):
        atributos_major, cobertura_major, _ = simulador.snapshots(configuracao, fontes_major)
    with patch.object(simulador, "base_v1", return_value=(jogos_tier1, series_tier1)):
        _, cobertura_major_ref, _ = simulador.snapshots(configuracao, fontes_major)

cobertura_fase1 = (
    cobertura_major.loc[cobertura_major["etapa_entrada"].eq(1)]
    .merge(
        cobertura_major_ref[["chave", "jogos_historico_local", "nome_local"]],
        on="chave", validate="one_to_one", suffixes=("_v3", "_ref"),
    )
    .sort_values("seed_entrada")
)
cobertura_antes = int(cobertura_fase1["nome_local_ref"].notna().sum())
cobertura_depois = int(cobertura_fase1["nome_local_v3"].notna().sum())
print(f"Histórico corroborado na Etapa 1 · referência {cobertura_antes}/16 · V3 {cobertura_depois}/16")
display(cobertura_fase1[[
    "nome_oficial", "nome_local_v3", "jogos_historico_local_ref",
    "jogos_historico_local_v3", "ranking_pontos",
]])

Histórico corroborado na Etapa 1 · referência 8/16 · V3 10/16


,nome_oficial,nome_local_v3,jogos_historico_local_ref,jogos_historico_local_v3,ranking_pontos
0,Ninjas in Pyjamas,Ninjas in Pyjamas,140,229,93
1,PVISION,PARIVISION,144,269,188
2,HEROIC,Heroic,223,274,76
3,FaZe Clan,FaZe Clan,242,268,90
4,Team Liquid,Team Liquid,203,229,128
5,Eternal Fire,Eternal Fire,113,216,76
6,100 Thieves,100 Thieves,0,47,55
7,9z Globant,9z Team,101,123,89
8,M80,M80,119,151,71
9,DENDELE,NaN,0,0,48


In [23]:
# Resolver as escalações arquivadas usando o histórico anterior à previsão
import re

CORTE_MAJOR = pd.Timestamp(configuracao["data_previsao"])
APELIDOS_MAJOR = {
    "naf-fly": "naf",
    "dev1ce": "device",
    "jottaaa": "jotta",
}


def nick_normalizado(texto):
    if pd.isna(texto):
        return None
    nick = str(texto).split("(", 1)[0].strip()
    nick = re.sub(r",\s*[A-Z]{2}$", "", nick).strip().casefold()
    return APELIDOS_MAJOR.get(nick, nick)


identidades = jogadores.loc[
    jogadores["dia"].lt(CORTE_MAJOR) & jogadores["nick_csv"].notna(),
    ["match_id", "lado", "id", "nick_csv", "dia"],
].copy()
identidades["nick"] = identidades["nick_csv"].map(nick_normalizado)
ids_por_nick = identidades.groupby("nick")["id"].agg(set).to_dict()

lados = pd.concat([
    series_v3[["match_id", f"team{lado}"]].rename(columns={f"team{lado}": "nome_equipe"})
    .assign(lado=lado)
    for lado in (1, 2)
], ignore_index=True)
identidades = identidades.merge(lados, on=["match_id", "lado"], validate="many_to_one")
recentes = identidades.loc[identidades["dia"].ge(CORTE_MAJOR - pd.Timedelta(days=90))]
ids_por_nick_e_time = recentes.groupby(["nick", "nome_equipe"])["id"].agg(set).to_dict()
nomes_locais = cobertura_major.set_index("chave")["nome_local"].to_dict()

# Substituições anunciadas para o Major antes do corte da previsão.
substituicoes_confirmadas = {
    "9z_globant": {"meyern": "history"},
    "fut_esports": {"cmtry": "SunPayus"},
    "astralis": {"jabbi": "Anlelele"},
    "vitality": {"jL": "apEX"},
    "legacy": {"n1ssim": "ksloks"},
}
escalas_major = []
for time in configuracao["participantes"]:
    chave = time["chave"]
    assert len(time["jogadores"]) == 5
    for inscrito in time["jogadores"]:
        jogador = substituicoes_confirmadas.get(chave, {}).get(inscrito, inscrito)
        nick = nick_normalizado(jogador)
        ids = ids_por_nick.get(nick, set())
        motivo = "apelido único no histórico"
        if len(ids) > 1:
            ids = ids & ids_por_nick_e_time.get((nick, nomes_locais[chave]), set())
            motivo = "apelido e equipe recente confirmada"
        jogador_id = next(iter(ids)) if len(ids) == 1 else None
        escalas_major.append({
            "chave": chave,
            "nome_oficial": time["nome_oficial"],
            "etapa_entrada": time["etapa_entrada"],
            "jogador": jogador,
            "id": jogador_id,
            "identificação": motivo if jogador_id is not None else "sem ID confirmado",
            "dia": CORTE_MAJOR,
        })
escalas_major = pd.DataFrame(escalas_major)
duplicados = escalas_major["id"].notna() & escalas_major["id"].duplicated(keep=False)
escalas_major.loc[duplicados, "id"] = np.nan
escalas_major.loc[duplicados, "identificação"] = "ID repetido em vagas do Major"
escalas_major["consulta_id"] = escalas_major.index
perfis_major = consultar_jogadores(escalas_major)
elencos_major = resumir_elenco(perfis_major, ["chave"])
atributos_v4 = {
    chave: {**atributos_major[chave], **medidas}
    for chave, medidas in elencos_major.set_index("chave")[
        ATRIBUTOS_FORMA + ATRIBUTOS_DIFICULDADE
    ].to_dict("index").items()
}
assert set(atributos_v4) == set(atributos_major)

cobertura_jogadores_major = (
    perfis_major.groupby(["chave", "nome_oficial", "etapa_entrada"], as_index=False)
    .agg(ids_confirmados=("id", "count"), forma_anterior=("adr_recente", "count"))
    .merge(elencos_major[["chave", "nivel_eventos_recente", "fracao_tier3_recente"]],
           on="chave", validate="one_to_one")
    .sort_values(["etapa_entrada", "nome_oficial"])
)
display(cobertura_jogadores_major.drop(columns="chave").round(3))

,nome_oficial,etapa_entrada,ids_confirmados,forma_anterior,nivel_eventos_recente,fracao_tier3_recente
0,100 Thieves,1,5,5,1.417,0.003
2,9z Globant,1,5,4,1.965,0.000
6,Chinggis Warriors,1,5,2,1.958,0.000
7,DENDELE,1,5,5,1.674,0.115
8,Eternal Fire,1,5,5,1.270,0.053
10,FaZe Clan,1,5,5,1.348,0.126
11,FlyQuest,1,5,4,1.928,0.000
15,GOSH,1,5,5,1.233,0.000
16,HEROIC,1,5,5,1.602,0.001
17,Imperial Esports,1,5,5,1.936,0.000


### 10.1 Chances de jogar cada mapa da abertura

A tabela apresenta, para os oito confrontos MD1, a probabilidade de cada mapa sair e de a equipe A vencer naquele mapa segundo o Elo. As diferenças agregadas alimentam a V5. Quando a equipe está sem identidade local corroborada, usamos o histórico geral para o veto e Elo inicial no mapa. As demais entradas da V4 seguem disponíveis.

In [24]:
# Probabilidades de séries do Major e detalhamento das chances na rodada inicial
from itertools import combinations

chaves = [time["chave"] for time in configuracao["participantes"]]
indice_equipe = {chave: i for i, chave in enumerate(chaves)}
nome_equipe = cobertura_major.set_index("chave")["nome_oficial"].to_dict()
nome_mapa = cobertura_major.set_index("chave")["nome_local"].to_dict()
chaves_etapa1 = configuracao["etapas"]["1"]
hoje_major = dia_num(CORTE_MAJOR)


def variaveis_confronto(chave_a, chave_b, formato):
    perfil_a, perfil_b = atributos_v4[chave_a], atributos_v4[chave_b]
    x = {
        coluna: (
            formato if coluna == "bestOf"
            else perfil_a[coluna.removeprefix("dif_")] - perfil_b[coluna.removeprefix("dif_")]
        ) for coluna in ENTRADAS_V4
    }
    com_veto, uniforme, distribuicao, elo = integrar_mapas(
        nome_mapa[chave_a], nome_mapa[chave_b], formato, hoje_major, MAPAS_MAJOR,
    )
    x.update(zip(COLUNAS_MAPAS, (2.0 * com_veto - 1.0, com_veto - uniforme), strict=True))
    return x, distribuicao, elo


matrizes = {}
pares = list(combinations(range(len(chaves)), 2))
for formato in FORMATOS:
    dados = pd.DataFrame([
        variaveis_confronto(chaves[a], chaves[b], formato)[0] for a, b in pares
    ], columns=ENTRADAS_V5)
    probs = modelo_v5.predict_proba(dados)[:, 1]
    matriz = np.full((len(chaves), len(chaves)), 0.5)
    for (a, b), p in zip(pares, probs, strict=True):
        matriz[a, b], matriz[b, a] = p, 1.0 - p
    assert np.isfinite(matriz).all() and ((matriz >= 0) & (matriz <= 1)).all()
    np.testing.assert_allclose(matriz + matriz.T, 1.0, atol=1e-12)
    matrizes[formato] = matriz
    ida = variaveis_confronto(chaves[0], chaves[1], formato)[0]
    volta = variaveis_confronto(chaves[1], chaves[0], formato)[0]
    teste_ordem = modelo_v5.predict_proba(pd.DataFrame([ida, volta], columns=ENTRADAS_V5))[:, 1]
    np.testing.assert_allclose(teste_ordem.sum(), 1.0, atol=1e-12)

resumo_abertura = []
mapas_abertura = []
for a, b in zip(chaves_etapa1[:8], chaves_etapa1[8:], strict=True):
    entrada, distribuicao, elo = variaveis_confronto(a, b, 1)
    p_v4 = modelo_v4.predict_proba(pd.DataFrame([entrada], columns=ENTRADAS_V4))[0, 1]
    i = int(distribuicao.argmax())
    resumo_abertura.append({
        "Equipe A": nome_equipe[a], "Equipe B": nome_equipe[b],
        "Mapa mais provável": MAPAS_MAJOR[i], "Chance do mapa (%)": 100 * distribuicao[i],
        "Vitória A V4 (%)": 100 * p_v4,
        "Vitória A V5 (%)": 100 * matrizes[1][indice_equipe[a], indice_equipe[b]],
    })
    for mapa, jogar, vencer in zip(MAPAS_MAJOR, distribuicao, elo, strict=True):
        mapas_abertura.append({
            "Equipe A": nome_equipe[a], "Equipe B": nome_equipe[b], "Mapa": mapa,
            "Sair do veto (%)": 100 * jogar, "Vitória A no mapa pelo Elo (%)": 100 * vencer,
        })
resumo_abertura = pd.DataFrame(resumo_abertura)
mapas_abertura = pd.DataFrame(mapas_abertura)
np.testing.assert_allclose(
    mapas_abertura.groupby("Equipe A")["Sair do veto (%)"].sum().to_numpy(), 100.0,
)
display(resumo_abertura.round(3))
display(mapas_abertura.round(3))

,Equipe A,Equipe B,Mapa mais provável,Chance do mapa (%),Vitória A V4 (%),Vitória A V5 (%)
0,Ninjas in Pyjamas,M80,Mirage,27.858,58.280,59.355
1,PVISION,DENDELE,Mirage,24.807,70.439,67.673
2,HEROIC,Imperial Esports,Mirage,21.234,68.172,69.456
3,FaZe Clan,Wildcard,Ancient,19.444,65.738,64.274
4,Team Liquid,GOSH,Mirage,28.142,68.645,70.332
5,Eternal Fire,FlyQuest,Ancient,24.443,54.818,57.311
6,100 Thieves,Chinggis Warriors,Nuke,21.934,61.194,64.093
7,9z Globant,Lynn Vision Gaming,Mirage,19.650,67.798,63.673


,Equipe A,Equipe B,Mapa,Sair do veto (%),Vitória A no mapa pelo Elo (%)
0,Ninjas in Pyjamas,M80,Ancient,19.475,55.644
1,Ninjas in Pyjamas,M80,Anubis,8.339,49.353
2,Ninjas in Pyjamas,M80,Cache,9.321,51.439
3,Ninjas in Pyjamas,M80,Dust2,13.323,49.847
4,Ninjas in Pyjamas,M80,Inferno,12.667,51.376
5,Ninjas in Pyjamas,M80,Mirage,27.858,52.909
6,Ninjas in Pyjamas,M80,Nuke,9.016,60.081
7,PVISION,DENDELE,Ancient,18.758,59.484
8,PVISION,DENDELE,Anubis,7.263,48.633
9,PVISION,DENDELE,Cache,8.636,47.289


### 10.2 Probabilidades por fase e campeão projetado

Rodamos **100.000 Majors**, com semente **25**. A tabela mostra, para os 32 times, a chance antes do evento de chegar a cada fase. Uma equipe com vaga direta começa com 100% nas etapas já garantidas. O favorito ao título é o time que vence mais simulações.

In [25]:
repeticoes = 100_000
semente = 25

campanhas = [(3, 0), (3, 1), (3, 2), (2, 3), (1, 3), (0, 3)]
indice_campanha = {campanha: i for i, campanha in enumerate(campanhas)}
equipe_etapa1 = [indice_equipe[chave] for chave in chaves_etapa1]
perfis_etapa1 = []
suico_original = simulador.swiss_stage


def suico_com_registro(equipes, etapa, matrizes, rng, prioridade, registrar):
    if etapa != 1:
        return suico_original(equipes, etapa, matrizes, rng, prioridade, registrar)
    series_jogadas = dict.fromkeys(equipes, 0)

    def registrar_partida(time_a, time_b, etapa_atual):
        series_jogadas[time_a] += 1
        series_jogadas[time_b] += 1
        registrar(time_a, time_b, etapa_atual)

    classificados = suico_original(
        equipes, etapa, matrizes, rng, prioridade, registrar_partida,
    )
    avancaram = set(classificados)
    perfil = []
    for equipe in equipe_etapa1:
        total = series_jogadas[equipe]
        vitorias = 3 if equipe in avancaram else total - 3
        perfil.append(indice_campanha[vitorias, total - vitorias])
    perfis_etapa1.append(tuple(perfil))
    return classificados


print("V5 · 100.000 simulações")
with patch.object(simulador, "swiss_stage", suico_com_registro):
    contagens, _, _ = simulador.monte_carlo(
        configuracao, chaves, matrizes, iterations=repeticoes, seed=semente,
    )
cenarios_etapa1 = np.asarray(perfis_etapa1, dtype=np.int8)
assert cenarios_etapa1.shape == (repeticoes, 16)
for i, vagas in enumerate((2, 3, 3, 3, 3, 2)):
    assert np.all((cenarios_etapa1 == i).sum(axis=1) == vagas)
np.testing.assert_array_equal(
    (cenarios_etapa1 < 3).sum(axis=0), contagens[equipe_etapa1, 0],
)

# Tabela com as probabilidades acumuladas por time e fase
fases = ["Etapa 2", "Etapa 3", "Quartas", "Semifinal", "Final", "Campeão", "3º lugar"]
equipes_major = cobertura_major.set_index("chave")[["nome_oficial", "etapa_entrada"]]
np.testing.assert_array_equal(contagens.sum(axis=0), repeticoes * np.array([24, 16, 8, 4, 2, 1, 1]))
chances_campeonato = (
    pd.DataFrame(100 * contagens / repeticoes, index=chaves, columns=fases)
    .join(equipes_major, validate="one_to_one")
    .sort_values("Campeão", ascending=False)
    .rename_axis("chave").reset_index()
)
chances_campeonato = chances_campeonato[["chave", "nome_oficial", "etapa_entrada", *fases]]
favorito = chances_campeonato.iloc[0]
favorito_oficial = favorito["nome_oficial"]
chave_favorito = favorito["chave"]
print(f"Favorito ao título · {favorito_oficial} · {favorito['Campeão']:.3f}%")
display(chances_campeonato.drop(columns="chave").round(3))

primeira_fase = (
    chances_campeonato.loc[chances_campeonato["etapa_entrada"].eq(1), ["chave", "nome_oficial", "Etapa 2"]]
    .rename(columns={"Etapa 2": "V5 (%)"})
    .sort_values("V5 (%)", ascending=False)
    .reset_index(drop=True)
)
display(primeira_fase[["nome_oficial", "V5 (%)"]].round(3))

V5 · 100.000 simulações
Simulacoes: 5000/100000
Simulacoes: 10000/100000
Simulacoes: 15000/100000
Simulacoes: 20000/100000
Simulacoes: 25000/100000
Simulacoes: 30000/100000
Simulacoes: 35000/100000
Simulacoes: 40000/100000
Simulacoes: 45000/100000
Simulacoes: 50000/100000
Simulacoes: 55000/100000
Simulacoes: 60000/100000
Simulacoes: 65000/100000
Simulacoes: 70000/100000
Simulacoes: 75000/100000
Simulacoes: 80000/100000
Simulacoes: 85000/100000
Simulacoes: 90000/100000
Simulacoes: 95000/100000
Simulacoes: 100000/100000
Favorito ao título · FURIA · 25.155%


,nome_oficial,etapa_entrada,Etapa 2,Etapa 3,Quartas,Semifinal,Final,Campeão,3º lugar
0,FURIA,3,100.000,100.000,92.611,64.431,41.123,25.155,15.503
1,MOUZ,3,100.000,100.000,92.018,62.583,38.992,24.051,15.518
2,Team Spirit,3,100.000,100.000,84.576,49.981,26.455,13.490,13.288
3,Team Vitality,3,100.000,100.000,82.500,46.594,23.600,11.225,12.401
4,Team Falcons,3,100.000,100.000,81.616,46.205,23.639,10.856,12.188
5,Aurora Gaming,3,100.000,100.000,75.467,38.696,17.768,7.130,10.418
6,Natus Vincere,2,100.000,91.165,57.349,26.532,11.250,4.178,6.824
7,The MongolZ,2,100.000,82.080,37.194,13.499,4.484,1.276,3.246
8,Legacy,3,100.000,100.000,37.844,10.921,2.930,0.636,2.355
9,FUT ESPORTS,2,100.000,74.623,24.420,7.300,1.959,0.468,1.631


,nome_oficial,V5 (%)
0,PVISION,82.886
1,9z Globant,73.850
2,HEROIC,71.987
3,FaZe Clan,71.972
4,Team Liquid,68.257
5,Eternal Fire,57.788
6,Ninjas in Pyjamas,57.297
7,Lynn Vision Gaming,48.248
8,100 Thieves,42.812
9,DENDELE,42.638


### 10.3 Passo a passo de um torneio completo

Reproduzimos a primeira simulação que termina com o título do favorito. O resultado de cada série vem desse mesmo torneio simulado. As chances por confronto valem quando os adversários da linha se encontram. A tabela de fases acima traz as chances antes de conhecer os adversários.

In [26]:
# Reproduzir um campeonato inteiro, com os mesmos sorteios do motor validado
from IPython.display import HTML


class SorteioRegistrado:
    def __init__(self, rng):
        self.rng = rng
        self.ultimo = None

    def random(self):
        self.ultimo = self.rng.random()
        return self.ultimo


def simular_trajetoria(rng):
    sorteio = SorteioRegistrado(rng)
    jogos_cenario = []
    fases_suicas = {
        etapa: [indice_equipe[chave] for chave in configuracao["etapas"][str(etapa)]]
        for etapa in (1, 2, 3)
    }
    prioridade = configuracao["tabela_prioridade_grupos_seis"]

    def adicionar(fase, rodada, a, b, formato, vencedor):
        jogos_cenario.append({
            "Fase": fase, "Rodada": rodada, "Formato": f"MD{formato}",
            "Equipe A": nome_equipe[chaves[a]], "Equipe B": nome_equipe[chaves[b]],
            "Vitória A (%)": 100 * matrizes[formato][a, b],
            "Vitória B (%)": 100 * matrizes[formato][b, a],
            "Vencedor no cenário": nome_equipe[chaves[vencedor]],
        })

    def rodada_suica(equipes, etapa):
        campanha = {equipe: [0, 0] for equipe in equipes}

        def registrar(a, b, _):
            vitorias, derrotas = campanha[a]
            assert campanha[b] == campanha[a]
            formato = 3 if etapa == 3 or vitorias == 2 or derrotas == 2 else 1
            vencedor, perdedor = (a, b) if sorteio.ultimo < matrizes[formato][a, b] else (b, a)
            adicionar(f"Etapa {etapa}", vitorias + derrotas + 1, a, b, formato, vencedor)
            campanha[vencedor][0] += 1
            campanha[perdedor][1] += 1

        classificados = simulador.swiss_stage(equipes, etapa, matrizes, sorteio, prioridade, registrar)
        assert set(classificados) == {equipe for equipe, recorde in campanha.items() if recorde[0] == 3}
        return classificados

    q1 = rodada_suica(fases_suicas[1], 1)
    q2 = rodada_suica(fases_suicas[2] + q1, 2)
    q3 = rodada_suica(fases_suicas[3] + q2, 3)

    def eliminatoria(fase, numero, a, b, formato):
        vencedor, perdedor = (a, b) if sorteio.random() < matrizes[formato][a, b] else (b, a)
        adicionar(fase, numero, a, b, formato, vencedor)
        return vencedor, perdedor

    semifinais = [
        eliminatoria("Quartas", i, q3[a - 1], q3[b - 1], 3)[0]
        for i, (a, b) in enumerate(configuracao["regras"]["quartas"], 1)
    ]
    finalista_a, perdeu_a = eliminatoria("Semifinal", 1, semifinais[0], semifinais[1], 3)
    finalista_b, perdeu_b = eliminatoria("Semifinal", 2, semifinais[2], semifinais[3], 3)
    terceiro, _ = eliminatoria("3º lugar", 1, perdeu_a, perdeu_b, 3)
    campeao, _ = eliminatoria("Final", 1, finalista_a, finalista_b, 5)
    marcos = [
        fases_suicas[2] + fases_suicas[3] + q1, fases_suicas[3] + q2,
        q3, semifinais, [finalista_a, finalista_b], [campeao], [terceiro],
    ]
    assert len(jogos_cenario) == 107
    return marcos, jogos_cenario


# A trilha deve reproduzir os totais do próprio motor em um prefixo da mesma semente.
controle, _, _ = simulador.monte_carlo(configuracao, chaves, matrizes, iterations=20, seed=semente)
espelho = np.zeros_like(controle)
rng_controle = np.random.default_rng(semente)
for _ in range(20):
    marcos, _ = simular_trajetoria(rng_controle)
    for coluna, equipes in enumerate(marcos):
        espelho[equipes, coluna] += 1
np.testing.assert_array_equal(espelho, controle)

rng = np.random.default_rng(semente)
for numero_cenario in range(1, repeticoes + 1):
    marcos, jogos_cenario = simular_trajetoria(rng)
    if chaves[marcos[5][0]] == chave_favorito:
        break
else:
    raise RuntimeError("A trajetória não encontrou o campeão registrado pela simulação.")

trajetoria_completa = pd.DataFrame(jogos_cenario)
final_do_cenario = trajetoria_completa.iloc[-1].to_dict()
caminho_favorito = trajetoria_completa.loc[
    trajetoria_completa["Equipe A"].eq(favorito_oficial)
    | trajetoria_completa["Equipe B"].eq(favorito_oficial)
].copy()
caminho_favorito["Adversário"] = np.where(
    caminho_favorito["Equipe A"].eq(favorito_oficial), caminho_favorito["Equipe B"], caminho_favorito["Equipe A"],
)
caminho_favorito["Vitória do favorito (%)"] = np.where(
    caminho_favorito["Equipe A"].eq(favorito_oficial), caminho_favorito["Vitória A (%)"], caminho_favorito["Vitória B (%)"],
)
caminho_favorito = caminho_favorito[[
    "Fase", "Rodada", "Formato", "Adversário", "Vitória do favorito (%)", "Vencedor no cenário",
]]
print(f"Trajetória {numero_cenario:,} · {favorito_oficial} campeão no cenário.")
display(caminho_favorito.round(3))
linhas = trajetoria_completa.round(3).to_html(index=False)
display(HTML(
    '<details><summary>Ver os 107 confrontos do cenário, da Etapa 1 à final</summary>'
    f'<div style="overflow-x:auto">{linhas}</div></details>'
))

Trajetória 1 · FURIA campeão no cenário.


,Fase,Rodada,Formato,Adversário,Vitória do favorito (%),Vencedor no cenário
68,Etapa 3,1,MD3,Team Liquid,88.432,FURIA
76,Etapa 3,2,MD3,Legacy,81.761,FURIA
83,Etapa 3,3,MD3,Team Vitality,60.994,Team Vitality
90,Etapa 3,4,MD3,The MongolZ,76.228,FURIA
102,Quartas,4,MD3,The MongolZ,76.228,FURIA
104,Semifinal,2,MD3,MOUZ,50.743,FURIA
106,Final,1,MD5,Team Falcons,62.582,FURIA


Fase,Rodada,Formato,Equipe A,Equipe B,Vitória A (%),Vitória B (%),Vencedor no cenário
Etapa 1,1,MD1,Ninjas in Pyjamas,M80,59.355,40.645,Ninjas in Pyjamas
Etapa 1,1,MD1,PVISION,DENDELE,67.673,32.327,PVISION
Etapa 1,1,MD1,HEROIC,Imperial Esports,69.456,30.544,HEROIC
Etapa 1,1,MD1,FaZe Clan,Wildcard,64.274,35.726,FaZe Clan
Etapa 1,1,MD1,Team Liquid,GOSH,70.332,29.668,Team Liquid
Etapa 1,1,MD1,Eternal Fire,FlyQuest,57.311,42.689,Eternal Fire
Etapa 1,1,MD1,100 Thieves,Chinggis Warriors,64.093,35.907,Chinggis Warriors
Etapa 1,1,MD1,9z Globant,Lynn Vision Gaming,63.673,36.327,Lynn Vision Gaming
Etapa 1,2,MD1,Ninjas in Pyjamas,Lynn Vision Gaming,46.581,53.419,Ninjas in Pyjamas
Etapa 1,2,MD1,PVISION,Chinggis Warriors,80.259,19.741,PVISION


## 11. Ler as campanhas da primeira fase em conjunto

Os mapas de calor mostram as chances individuais e as chances conjuntas das duplas 3–0 e 0–3. Ambos usam as 100.000 simulações da tabela do campeonato. O mapa de duplas também abre com as contagens salvas em um kernel recém-iniciado.

In [27]:
# Gráfico 1 — chances individuais por campanha na Etapa 1
from html import escape

from IPython.display import HTML

ordem_pelo_resumo = {chave: i for i, chave in enumerate(chaves_etapa1)}
ordem = primeira_fase["chave"].map(ordem_pelo_resumo).to_numpy()
cenarios = cenarios_etapa1[:, ordem]
nomes = primeira_fase["nome_oficial"].tolist()
placares = [f"{vitorias}–{derrotas}" for vitorias, derrotas in campanhas]
contagem_campanhas = np.column_stack(
    [(cenarios == codigo).sum(axis=0) for codigo in range(len(campanhas))]
)
avancos = contagem_campanhas[:, :3].sum(axis=1)
np.testing.assert_array_equal(contagem_campanhas.sum(axis=1), np.full(len(nomes), repeticoes))
np.testing.assert_array_equal(100 * avancos / repeticoes, primeira_fase["V5 (%)"].to_numpy())


def inteiro_pt(numero):
    return f"{numero:,}".replace(",", ".")


def percentual(ocorrencias):
    return f"{100 * ocorrencias / repeticoes:.3f}%".replace(".", ",")


amostra = f"V5 · {inteiro_pt(repeticoes)} simulações · semente {semente}"
estilo_painel = "background:#fff;color:#172a34;padding:8px 4px;font:14px system-ui,sans-serif"
estilo_tabela = (
    "background:#fff;color:#172a34;border-collapse:separate;border-spacing:3px;"
    "font:13px system-ui,sans-serif;font-variant-numeric:tabular-nums;text-align:center"
)
verde = "99,188,151"
roxo = "170,147,215"

cabecalho = "".join(
    f'<th scope="col" style="padding:6px 10px">{placar}</th>' for placar in placares
)
linhas = []
maximos = contagem_campanhas.max(axis=0)
teto = maximos.max()
destaques = []
for coluna in (0, 5):
    lideres = ", ".join(
        escape(nome) for nome, vezes in zip(nomes, contagem_campanhas[:, coluna], strict=True)
        if vezes == maximos[coluna]
    )
    destaques.append(f'<strong>{placares[coluna]}</strong> · {lideres} ({percentual(maximos[coluna])})')

for nome, valores, avanco in zip(nomes, contagem_campanhas, avancos, strict=True):
    celulas = []
    for coluna, ocorrencias in enumerate(valores):
        rgb = verde if coluna < 3 else roxo
        intensidade = 0.12 + 0.88 * ocorrencias / teto
        estilo = f"background:rgba({rgb},{intensidade:.3f});padding:5px 9px;border-radius:4px"
        if coluna in (0, 5) and ocorrencias == maximos[coluna]:
            estilo += ";font-weight:750;outline:2px solid #374c58;outline-offset:-2px"
        celulas.append(f'<td style="{estilo}">{percentual(ocorrencias)}</td>')
    celulas.append(
        f'<td style="background:#e0f2ea;padding:5px 9px;font-weight:700">{percentual(avanco)}</td>'
    )
    linhas.append(
        '<tr><th scope="row" style="text-align:left;padding:5px 10px;white-space:nowrap">'
        f'{escape(nome)}</th>{"".join(celulas)}</tr>'
    )

display(HTML(f"""
<div style="{estilo_painel}">
<h3 style="margin:0 0 4px">1. Chances por time</h3>
<p style="margin:0 0 10px">{amostra}</p>
<p style="margin:0 0 8px">{" &nbsp;|&nbsp; ".join(destaques)}</p>
<div style="overflow-x:auto">
<table style="{estilo_tabela};min-width:720px">
    <thead>
        <tr>
            <th rowspan="2" scope="col" style="text-align:left;padding:6px 10px">Equipe</th>
            <th colspan="3" scope="colgroup" style="background:#dcefe7;padding:6px 10px">Classificação</th>
            <th colspan="3" scope="colgroup" style="background:#ece4f5;padding:6px 10px">Eliminação</th>
            <th rowspan="2" scope="col" style="background:#dcefe7;padding:6px 10px">Avança (V5)</th>
        </tr>
        <tr>{cabecalho}</tr>
    </thead>
    <tbody>{"".join(linhas)}</tbody>
</table>
</div>
</div>
"""))

In [28]:
# Gráfico 2 — duplas 3–0 e 0–3 no mesmo cenário
from collections import Counter
from html import escape
from itertools import combinations

from IPython.display import HTML, display

requisitos = {"cenarios_etapa1", "chaves_etapa1", "primeira_fase", "repeticoes", "semente"}
usando_simulacao = requisitos <= globals().keys()
if usando_simulacao:
    import numpy as np

    colunas_por_chave = {chave: i for i, chave in enumerate(chaves_etapa1)}
    ordem_duplas = primeira_fase["chave"].map(colunas_por_chave).to_numpy()
    perfis_duplas = cenarios_etapa1[:, ordem_duplas]
    nomes = primeira_fase["nome_oficial"].tolist()
    duplas_3_0 = map(tuple, np.where(perfis_duplas == 0)[1].reshape(-1, 2))
    duplas_0_3 = map(tuple, np.where(perfis_duplas == 5)[1].reshape(-1, 2))
    extremos_da_simulacao = Counter(zip(duplas_3_0, duplas_0_3, strict=True))
    contagem_3_0 = Counter()
    contagem_0_3 = Counter()
    for (invictos, sem_vitorias), quantidade in extremos_da_simulacao.items():
        contagem_3_0[invictos] += quantidade
        contagem_0_3[sem_vitorias] += quantidade
    extremos_mais_frequentes = extremos_da_simulacao.most_common(5)
else:
    # Contagens da execução salva, para abrir o mapa mesmo em um kernel novo.
    repeticoes, semente = 100_000, 25
    nomes = ['PVISION', '9z Globant', 'HEROIC', 'FaZe Clan', 'Team Liquid', 'Eternal Fire', 'Ninjas in Pyjamas', 'Lynn Vision Gaming', '100 Thieves', 'DENDELE', 'M80', 'Wildcard', 'FlyQuest', 'GOSH', 'Imperial Esports', 'Chinggis Warriors']
    pares = list(combinations(range(len(nomes)), 2))
    salvas_3_0 = [
        3838, 5340, 4942, 4271, 2848, 4033, 1105, 1427, 0, 1263, 887, 759, 437, 555, 304,
        2722, 2526, 2466, 1710, 1741, 0, 915, 938, 668, 817, 1015, 443, 430, 548, 3078,
        2928, 1703, 2487, 981, 937, 920, 807, 587, 641, 307, 0, 292, 2375, 1806, 2096, 1029,
        820, 816, 772, 0, 713, 347, 313, 339, 1274, 1867, 1188, 928, 839, 738, 609, 720, 0,
        335, 360, 1105, 908, 480, 545, 505, 471, 0, 217, 255, 241, 0, 493, 647, 0, 412, 301,
        172, 282, 77, 643, 468, 395, 416, 384, 222, 203, 241, 298, 292, 302, 391, 156, 156,
        0, 251, 260, 276, 142, 130, 162, 255, 306, 136, 128, 139, 247, 132, 112, 134, 153,
        134, 126, 75, 64, 60,
    ]
    contagem_3_0 = Counter(dict(zip(pares, salvas_3_0, strict=True)))
    salvas_0_3 = [
        64, 69, 73, 78, 141, 143, 61, 193, 0, 263, 235, 174, 348, 431, 231, 108, 97, 92,
        174, 178, 0, 223, 275, 212, 273, 301, 493, 495, 600, 143, 140, 246, 242, 174, 330,
        474, 468, 414, 342, 573, 0, 415, 121, 230, 235, 241, 288, 404, 405, 0, 309, 567,
        619, 583, 218, 257, 296, 331, 395, 414, 386, 350, 0, 705, 667, 433, 521, 477, 653,
        641, 639, 0, 903, 1085, 1201, 0, 524, 913, 0, 897, 660, 1216, 1565, 954, 802, 963,
        1031, 1347, 1491, 1994, 1972, 3177, 760, 817, 928, 969, 1328, 1472, 0, 1100, 1405,
        1298, 1801, 1867, 2201, 1276, 1224, 1906, 2181, 2096, 1792, 2223, 2131, 3027, 2650,
        2414, 3480, 3952, 4870, 4766,
    ]
    contagem_0_3 = Counter(dict(zip(pares, salvas_0_3, strict=True)))
    extremos_mais_frequentes = [(((0, 2), (14, 15)), 307), (((2, 4), (13, 14)), 265), (((0, 3), (13, 15)), 258), (((0, 2), (13, 15)), 253), (((0, 2), (13, 14)), 243)]

assert sum(contagem_3_0.values()) == sum(contagem_0_3.values()) == repeticoes


def inteiro_pt(numero):
    return f"{numero:,}".replace(",", ".")


def percentual(ocorrencias):
    return f"{100 * ocorrencias / repeticoes:.3f}%".replace(".", ",")


amostra = f"V5 · {inteiro_pt(repeticoes)} simulações · semente {semente}"
if not usando_simulacao:
    amostra += " · resultados salvos"
estilo_painel = "background:#fff;color:#172a34;padding:8px 4px;font:14px system-ui,sans-serif"
estilo_tabela = (
    "background:#fff;color:#172a34;border-collapse:separate;border-spacing:3px;"
    "font:13px system-ui,sans-serif;font-variant-numeric:tabular-nums;text-align:center"
)
verde = "99,188,151"
roxo = "170,147,215"


def nome_da_dupla(dupla):
    return " + ".join(nomes[i] for i in dupla)


def mapa_de_duplas(contagem, placar, cor):
    maior = max(contagem.values())
    lideres = [nome_da_dupla(dupla) for dupla, vezes in contagem.items() if vezes == maior]
    lideres_html = " / ".join(escape(nome) for nome in lideres)
    titulo = "Dupla mais frequente" if len(lideres) == 1 else "Duplas empatadas na maior frequência"
    cabecalho = "".join(
        f'<th scope="col" style="min-width:58px;max-width:80px;padding:6px 3px;'
        f'font-size:11px;line-height:1.2">{escape(nome)}</th>'
        for nome in nomes
    )
    linhas = []
    for i, nome in enumerate(nomes):
        linha = [
            f'<th scope="row" style="text-align:left;white-space:nowrap;padding:4px 8px">'
            f'{escape(nome)}</th>'
        ]
        for j in range(len(nomes)):
            if j <= i:
                linha.append('<td style="background:#f5f7f7"></td>')
                continue
            dupla = (i, j)
            vezes = contagem[dupla]
            intensidade = 0.08 + 0.82 * vezes / maior
            estilo = (
                f"background-color:rgba({cor},{intensidade:.3f});"
                "font-size:11px;padding:5px 3px;white-space:nowrap"
            )
            if vezes == maior:
                estilo += ";font-weight:750;outline:2px solid #374c58;outline-offset:-2px"
            dica = escape(f"{nome_da_dupla(dupla)} · {inteiro_pt(vezes)} simulações")
            linha.append(f'<td style="{estilo}" title="{dica}">{percentual(vezes)}</td>')
        linhas.append(f'<tr>{"".join(linha)}</tr>')

    return f"""
    <section style="margin:14px 0">
        <h4 style="margin:0 0 4px">Ambos {placar}</h4>
        <p style="margin:4px 0 8px">{titulo} · <strong>{lideres_html}</strong>
            · <strong>{percentual(maior)}</strong> ({inteiro_pt(maior)} simulações)</p>
        <div style="overflow-x:auto">
        <table style="{estilo_tabela}" aria-label="Chance dos dois times terminarem {placar}">
            <thead><tr><th scope="col" style="text-align:left;padding:6px 8px">Equipe</th>{cabecalho}</tr></thead>
            <tbody>{"".join(linhas)}</tbody>
        </table>
        </div>
    </section>
    """


linhas_extremos = []
for (invictos, sem_vitorias), vezes in extremos_mais_frequentes:
    linhas_extremos.append(
        f'<tr><td style="background:#e5f4ed;padding:6px 10px">{escape(nome_da_dupla(invictos))}</td>'
        f'<td style="background:#eee8f7;padding:6px 10px">{escape(nome_da_dupla(sem_vitorias))}</td>'
        f'<td style="padding:6px 10px">{inteiro_pt(vezes)}</td>'
        f'<td style="font-weight:700;padding:6px 10px">{percentual(vezes)}</td></tr>'
    )

dupla_exemplo = tuple(
    i for i, nome in enumerate(nomes) if nome in {"PVISION", "Team Liquid"}
)
exemplo_html = ""
if len(dupla_exemplo) == 2:
    vezes_exemplo = contagem_3_0[dupla_exemplo]
    exemplo_html = (
        f'<p><strong>{escape(nome_da_dupla(dupla_exemplo))}</strong> · ambos 3–0 '
        f'em {percentual(vezes_exemplo)} ({inteiro_pt(vezes_exemplo)} simulações)</p>'
    )

display(HTML(f"""
<div style="{estilo_painel}">
    <h3 style="margin:0 0 4px">2. Probabilidades conjuntas por dupla</h3>
    <p style="margin:0 0 10px">{amostra}</p>
    {mapa_de_duplas(contagem_3_0, "3–0", verde)}
    {exemplo_html}
    {mapa_de_duplas(contagem_0_3, "0–3", roxo)}
    <h4 style="margin:18px 0 4px">Duplas 3–0 e 0–3 que mais saíram juntas</h4>
    <div style="overflow-x:auto">
    <table style="{estilo_tabela}">
        <thead><tr><th scope="col">Ambos 3–0</th><th scope="col">Ambos 0–3</th>
        <th scope="col">Simulações</th><th scope="col">Chance conjunta</th></tr></thead>
        <tbody>{"".join(linhas_extremos)}</tbody>
    </table>
    </div>
</div>
"""))

Equipe,PVISION,9z Globant,HEROIC,FaZe Clan,Team Liquid,Eternal Fire,Ninjas in Pyjamas,Lynn Vision Gaming,100 Thieves,DENDELE,M80,Wildcard,FlyQuest,GOSH,Imperial Esports,Chinggis Warriors
PVISION,,"3,838%","5,340%","4,942%","4,271%","2,848%","4,033%","1,105%","1,427%","0,000%","1,263%","0,887%","0,759%","0,437%","0,555%","0,304%"
9z Globant,,,"2,722%","2,526%","2,466%","1,710%","1,741%","0,000%","0,915%","0,938%","0,668%","0,817%","1,015%","0,443%","0,430%","0,548%"
HEROIC,,,,"3,078%","2,928%","1,703%","2,487%","0,981%","0,937%","0,920%","0,807%","0,587%","0,641%","0,307%","0,000%","0,292%"
FaZe Clan,,,,,"2,375%","1,806%","2,096%","1,029%","0,820%","0,816%","0,772%","0,000%","0,713%","0,347%","0,313%","0,339%"
Team Liquid,,,,,,"1,274%","1,867%","1,188%","0,928%","0,839%","0,738%","0,609%","0,720%","0,000%","0,335%","0,360%"
Eternal Fire,,,,,,,"1,105%","0,908%","0,480%","0,545%","0,505%","0,471%","0,000%","0,217%","0,255%","0,241%"
Ninjas in Pyjamas,,,,,,,,"0,000%","0,493%","0,647%","0,000%","0,412%","0,301%","0,172%","0,282%","0,077%"
Lynn Vision Gaming,,,,,,,,,"0,643%","0,468%","0,395%","0,416%","0,384%","0,222%","0,203%","0,241%"
100 Thieves,,,,,,,,,,"0,298%","0,292%","0,302%","0,391%","0,156%","0,156%","0,000%"
DENDELE,,,,,,,,,,,"0,251%","0,260%","0,276%","0,142%","0,130%","0,162%"
